# Original NutriVision Research Notebook — Source-only Archive

This notebook is a **cleaned historical source archive** from the user-provided `NutriVision_AI.ipynb` (133 original code cells). Cell source code is retained **verbatim**, with its original 0-based index recorded in each code cell's metadata. Execution outputs, saved pictures, widget state, execution counts and machine-specific metadata have been removed for privacy, size, and academic clarity.

**Do not use Run All:** this is a chronological research log with earlier baselines, later revisions, Colab-only commands, multiple runtime resets and intermediate Drive checkpoints. It is evidence of original experiments, not an independently validated end-to-end training pipeline. For stage-specific extracts, use the other notebooks in this folder. Reported historical results must be distinguished from freshly reproduced runs.

The original notebook includes the training code that saved `convnext_stage3_best.pth`, `dinov2_last4_best.pth`, and `nutrition_mlp_best.pth`. It does **not** contain the Food Presence V2 head training routine.


## Initial Food-101 baseline CNN — original cell 0

In [ ]:

import torch

print("PyTorch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Selected device:", device)

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))


In [ ]:

import torch
import torchvision
import shutil

from torchvision import datasets

# Check available disk space
total, used, free = shutil.disk_usage("/content")

print(f"Free storage: {free / (1024**3):.2f} GB")

# Download Food-101
train_data = datasets.Food101(
    root="/content/data",
    split="train",
    download=True
)

test_data = datasets.Food101(
    root="/content/data",
    split="test",
    download=False
)

print("Training images:", len(train_data))
print("Testing images:", len(test_data))
print("Number of classes:", len(train_data.classes))
print("First 10 classes:", train_data.classes[:10])


In [ ]:

import matplotlib.pyplot as plt
import random

random.seed(42)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax in axes.flat:
    index = random.randrange(len(train_data))

    image, label = train_data[index]

    ax.imshow(image)
    ax.set_title(train_data.classes[label])
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

train_processed = datasets.Food101(
    root="/content/data",
    split="train",
    transform=transform,
    download=False
)

train_loader = DataLoader(
    train_processed,
    batch_size=16,
    shuffle=True,
    num_workers=2
)

images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Pixel range:", images.min().item(), images.max().item())


In [ ]:

from torch.utils.data import random_split, DataLoader

# Total training samples
total_size = len(train_processed)

# 10% for validation
val_size = int(total_size * 0.10)
train_size = total_size - val_size

# Reproducible random split
generator = torch.Generator().manual_seed(42)

train_subset, val_subset = random_split(
    train_processed,
    [train_size, val_size],
    generator=generator
)

train_loader = DataLoader(
    train_subset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training:", len(train_subset))
print("Validation:", len(val_subset))


In [ ]:

import torch.nn as nn

class FoodCNN(nn.Module):

    def __init__(self, num_classes=101):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = FoodCNN(num_classes=101).to(device)

print(model)


In [ ]:

# Create two fake images to test the architecture
dummy_input = torch.randn(
    2, 3, 128, 128,
    device=device
)

model.eval()

with torch.no_grad():
    output = model(dummy_input)

print("Input shape:", dummy_input.shape)
print("Output shape:", output.shape)

total_params = sum(
    p.numel() for p in model.parameters()
)

print("Total parameters:", total_params)


In [ ]:

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Enable training mode
model.train()

# Get one batch of real food images
images, labels = next(iter(train_loader))

# Move data to GPU
images = images.to(device)
labels = labels.to(device)

# Clear old gradients
optimizer.zero_grad()

# 1. Forward Pass
outputs = model(images)

# 2. Calculate Loss
loss = criterion(outputs, labels)

print("Loss before update:", loss.item())

# 3. Backpropagation
loss.backward()

# Check gradient of first Conv layer
gradient = model.features[0].weight.grad

print("Mean absolute gradient:",
      gradient.abs().mean().item())

# Save weights before update
old_weights = model.features[0].weight.detach().clone()

# 4. Update Weights
optimizer.step()

# Verify that the weights changed
weight_change = (
    model.features[0].weight.detach() - old_weights
).abs().mean().item()

print("Mean weight change:", weight_change)


In [ ]:

from google.colab import drive
import os

drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/NutriVision_AI"
os.makedirs(save_dir, exist_ok=True)

print("Save directory:", save_dir)


In [ ]:

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Start a fresh, reproducible experiment
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

model = FoodCNN(num_classes=101).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 3

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

def run_epoch(loader, training=True):

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for images, labels in tqdm(loader, leave=False):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):

            # Forward Pass
            outputs = model(images)

            # Calculate Loss
            loss = criterion(outputs, labels)

            if training:
                # Backpropagation
                loss.backward()

                # Update Weights
                optimizer.step()

        # Calculate statistics
        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size

        predictions = outputs.argmax(dim=1)

        total_correct += (
            predictions == labels
        ).sum().item()

        total_samples += batch_size

    avg_loss = total_loss / total_samples
    accuracy = 100 * total_correct / total_samples

    return avg_loss, accuracy


best_val_acc = -1

for epoch in range(epochs):

    train_loss, train_acc = run_epoch(
        train_loader,
        training=True
    )

    val_loss, val_acc = run_epoch(
        val_loader,
        training=False
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)

    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"\nEpoch {epoch + 1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "val_accuracy": val_acc,
            "classes": train_processed.classes
        }, f"{save_dir}/best_foodcnn.pth")

        print("Best model saved!")

print("\nTraining completed!")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")


In [ ]:

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)

plt.plot(history["train_loss"], label="Training Loss")
plt.plot(history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

plt.subplot(1, 2, 2)

plt.plot(history["train_acc"], label="Training Accuracy")
plt.plot(history["val_acc"], label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training vs Validation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:

train_loader = DataLoader(
    train_subset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("DataLoaders ready!")


In [ ]:

# Load our best trained model
checkpoint = torch.load(
    f"{save_dir}/best_foodcnn.pth",
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Get food images
images, labels = next(iter(val_loader))

# Predict food classes
with torch.no_grad():
    outputs = model(images.to(device))
    predictions = outputs.argmax(dim=1).cpu()

# Show 4 predictions
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i in range(4):
    axes[i].imshow(images[i].permute(1, 2, 0))

    actual = train_processed.classes[labels[i].item()]
    predicted = train_processed.classes[predictions[i].item()]

    axes[i].set_title(
        f"Actual: {actual}\nPredicted: {predicted}",
        fontsize=9
    )
    axes[i].axis("off")

plt.tight_layout()
plt.show()


## ResNet18 transfer-learning baselines — original cell 13

In [ ]:

from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn

# Load a pretrained ResNet18
weights = ResNet18_Weights.DEFAULT
resnet_model = resnet18(weights=weights)

print("Old output layer:", resnet_model.fc)

# Replace its final classification layer
resnet_model.fc = nn.Linear(512, 101)

# Move model to GPU
resnet_model = resnet_model.to(device)

print("New output layer:", resnet_model.fc)


In [ ]:

from torchvision import transforms

# Prepare training images
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Prepare validation images
val_transform = weights.transforms()

print("Image transformations ready!")


In [ ]:

from torchvision import datasets
from torch.utils.data import DataLoader, Subset

# Load the same Food-101 images
train_data_resnet = datasets.Food101(
    root="/content/data",
    split="train",
    transform=train_transform,
    download=False
)

val_data_resnet = datasets.Food101(
    root="/content/data",
    split="train",
    transform=val_transform,
    download=False
)

# Keep the same train/validation split
resnet_train = Subset(
    train_data_resnet,
    train_subset.indices
)

resnet_val = Subset(
    val_data_resnet,
    val_subset.indices
)

# Create batches
resnet_train_loader = DataLoader(
    resnet_train,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

resnet_val_loader = DataLoader(
    resnet_val,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

images, labels = next(iter(resnet_train_loader))

print("Image shape:", images.shape)
print("Training:", len(resnet_train))
print("Validation:", len(resnet_val))


In [ ]:

# Freeze all pretrained layers
for param in resnet_model.parameters():
    param.requires_grad = False

# Allow the last layer to learn
for param in resnet_model.fc.parameters():
    param.requires_grad = True

# Loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    resnet_model.fc.parameters(),
    lr=0.001
)

print("ResNet18 is ready for training!")


In [ ]:

from tqdm.auto import tqdm

# Keep pretrained BatchNorm statistics fixed
resnet_model.eval()

total_loss = 0
correct = 0
total = 0

for images, labels in tqdm(resnet_train_loader):

    # Send images to GPU
    images = images.to(device)
    labels = labels.to(device)

    # Clear previous gradients
    optimizer.zero_grad()

    # 1. Forward pass
    outputs = resnet_model(images)

    # 2. Calculate loss
    loss = criterion(outputs, labels)

    # 3. Backpropagation
    loss.backward()

    # 4. Update final layer
    optimizer.step()

    # Track results
    total_loss += loss.item() * labels.size(0)
    correct += (outputs.argmax(1) == labels).sum().item()
    total += labels.size(0)

print("Train Loss:", round(total_loss / total, 4))
print("Train Accuracy:", round(100 * correct / total, 2), "%")


In [ ]:

resnet_model.eval()

correct = 0
total = 0
total_loss = 0

with torch.no_grad():

    for images, labels in tqdm(resnet_val_loader):

        images = images.to(device)
        labels = labels.to(device)

        outputs = resnet_model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

val_acc = 100 * correct / total

print("Validation Loss:", round(total_loss / total, 4))
print("Validation Accuracy:", round(val_acc, 2), "%")

# Save the trained model
torch.save(
    resnet_model.state_dict(),
    f"{save_dir}/resnet18_epoch1.pth"
)

print("Model saved!")
x

In [ ]:

import os

model_path = f"{save_dir}/resnet18_epoch1.pth"

if os.path.exists(model_path):
    print("Model saved successfully!")
    print("File size:",
          round(os.path.getsize(model_path) / (1024**2), 2),
          "MB")
else:
    print("Model file not found!")


In [ ]:

# 1. Load the saved model
path = f"{save_dir}/resnet18_epoch1.pth"

resnet_model.load_state_dict(
    torch.load(path, map_location=device, weights_only=True)
)

resnet_model.eval()

# 2. Get validation images
images, labels = next(iter(resnet_val_loader))

# 3. Make predictions
with torch.no_grad():
    outputs = resnet_model(images.to(device))
    predictions = outputs.argmax(dim=1).cpu()

# 4. Restore image colors for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# 5. Display 4 predictions
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i in range(4):
    image = (images[i] * std + mean).clamp(0, 1)

    actual = train_data_resnet.classes[labels[i].item()]
    predicted = train_data_resnet.classes[predictions[i].item()]

    axes[i].imshow(image.permute(1, 2, 0))
    axes[i].set_title(
        f"Actual: {actual}\nPredicted: {predicted}",
        fontsize=9
    )
    axes[i].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import torch
import os

# 1. Save train/validation split
torch.save({
    "train_indices": train_subset.indices,
    "val_indices": val_subset.indices
}, f"{save_dir}/data_split.pth")

# 2. Save ResNet18 training checkpoint
torch.save({
    "epoch": 1,
    "model_state_dict": resnet_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "train_accuracy": 44.12,
    "val_accuracy": 52.20
}, f"{save_dir}/resnet18_checkpoint.pth")

# 3. Check saved files
files = [
    "best_foodcnn.pth",
    "resnet18_epoch1.pth",
    "resnet18_checkpoint.pth",
    "data_split.pth"
]

for filename in files:
    path = os.path.join(save_dir, filename)
    if os.path.exists(path):
        print("SAVED:", filename)
    else:
        print("MISSING:", filename)


In [ ]:
import os

dataset_path = "/content/data/food-101"

if os.path.exists(dataset_path):
    print("✅ Dataset موجودة، ما تحتاج تحملها")
else:
    print("❌ Dataset غير موجودة، تحتاج تحميلها")

In [ ]:

import os
import torch
from google.colab import drive

# Connect Google Drive
drive.mount("/content/drive")

# Our project folder
save_dir = "/content/drive/MyDrive/NutriVision_AI"

# Check saved models
files = [
    "best_foodcnn.pth",
    "resnet18_checkpoint.pth",
    "resnet18_epoch1.pth",
    "data_split.pth"
]

for filename in files:
    path = os.path.join(save_dir, filename)

    if os.path.exists(path):
        print("SAVED:", filename)
    else:
        print("MISSING:", filename)

# Check Food-101 dataset
dataset_path = "/content/data/food-101"

if os.path.exists(dataset_path):
    print("\nDataset folder exists")
else:
    print("\nDataset needs downloading")


In [ ]:

from torchvision import datasets

train_data = datasets.Food101(
    root="/content/data",
    split="train",
    download=True
)

print("Dataset ready!")


In [ ]:

import torch
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset
from google.colab import drive

drive.mount("/content/drive")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_dir = "/content/drive/MyDrive/NutriVision_AI"

# Restore the same data split
split = torch.load(
    f"{save_dir}/data_split.pth",
    weights_only=True
)

weights = ResNet18_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

val_transform = weights.transforms()

train_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=train_transform,
    download=True
)

val_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=val_transform,
    download=False
)

train_subset = Subset(train_data, split["train_indices"])
val_subset = Subset(val_data, split["val_indices"])

train_loader = DataLoader(
    train_subset, batch_size=32,
    shuffle=True, num_workers=0
)

val_loader = DataLoader(
    val_subset, batch_size=32,
    shuffle=False, num_workers=0
)

print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))
print("Device:", device)


In [ ]:

import torch.nn as nn

# Create ResNet18 architecture
resnet_model = resnet18(weights=None)

# Set 101 food classes
resnet_model.fc = nn.Linear(512, 101)

# Load our previously trained weights
resnet_model.load_state_dict(
    torch.load(
        f"{save_dir}/resnet18_epoch1.pth",
        map_location="cpu",
        weights_only=True
    )
)

resnet_model = resnet_model.to(device)

# Unfreeze ALL layers
for param in resnet_model.parameters():
    param.requires_grad = True

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer for ALL layers
optimizer = torch.optim.Adam(
    resnet_model.parameters(),
    lr=0.00001
)

print("All layers are trainable!")


In [ ]:

from tqdm.auto import tqdm

# Training mode
resnet_model.train()

train_correct = 0
train_total = 0
train_loss_sum = 0

for images, labels in tqdm(train_loader, desc="Training"):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = resnet_model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    train_loss_sum += loss.item() * labels.size(0)
    train_correct += (outputs.argmax(1) == labels).sum().item()
    train_total += labels.size(0)

train_acc = 100 * train_correct / train_total

print("Train Loss:", round(train_loss_sum / train_total, 4))
print("Train Accuracy:", round(train_acc, 2), "%")

# Validation mode
resnet_model.eval()

val_correct = 0
val_total = 0
val_loss_sum = 0

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Validation"):

        images = images.to(device)
        labels = labels.to(device)

        outputs = resnet_model(images)
        loss = criterion(outputs, labels)

        val_loss_sum += loss.item() * labels.size(0)
        val_correct += (outputs.argmax(1) == labels).sum().item()
        val_total += labels.size(0)

val_acc = 100 * val_correct / val_total

print("Validation Loss:", round(val_loss_sum / val_total, 4))
print("Validation Accuracy:", round(val_acc, 2), "%")

# Save this new experiment
torch.save({
    "model_state_dict": resnet_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "train_accuracy": train_acc,
    "val_accuracy": val_acc,
    "epoch": 2
}, f"{save_dir}/resnet18_fullfinetune_epoch2.pth")

print("Full fine-tuning checkpoint saved!")


In [ ]:

from tqdm.auto import tqdm

def run_resnet_epoch(loader, training):

    if training:
        resnet_model.train()
    else:
        resnet_model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.set_grad_enabled(training):

        for images, labels in tqdm(loader):

            images = images.to(device)
            labels = labels.to(device)

            if training:
                optimizer.zero_grad()

            outputs = resnet_model(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, 100 * correct / total


# Restore the saved end of Epoch 2
checkpoint = torch.load(
    f"{save_dir}/resnet18_fullfinetune_epoch2.pth",
    map_location=device,
    weights_only=True
)

resnet_model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

# Train Epoch 3
train_loss, train_acc = run_resnet_epoch(
    train_loader, training=True
)

# Validate Epoch 3
val_loss, val_acc = run_resnet_epoch(
    val_loader, training=False
)

print("Epoch 3 Results")
print(f"Train Loss: {train_loss:.4f}")
print(f"Train Accuracy: {train_acc:.2f}%")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.2f}%")

# Save without overwriting Epoch 2
torch.save({
    "epoch": 3,
    "model_state_dict": resnet_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "train_accuracy": train_acc,
    "val_accuracy": val_acc,
    "val_loss": val_loss
}, f"{save_dir}/resnet18_fullfinetune_epoch3.pth")

print("Epoch 3 saved!")


In [ ]:
import os

path = f"{save_dir}/resnet18_fullfinetune_epoch3.pth"

print("Epoch 3 saved:", os.path.exists(path))

## Nutrition5k / ResNet18 calorie-regression experiments — original cell 30

In [ ]:

# Download nutritional labels only
!gsutil cp gs://nutrition5k_dataset/nutrition5k_dataset/metadata/dish_metadata_cafe1.csv /content/


In [ ]:

import pandas as pd

# Names of the columns we need
columns = [
    "dish_id",
    "calories_kcal",
    "mass_g",
    "fat_g",
    "carbs_g",
    "protein_g"
]

# Read the first 6 columns
df = pd.read_csv(
    "/content/dish_metadata_cafe1.csv",
    header=None,
    usecols=range(6),
    names=columns
)

print("Number of dishes:", len(df))

# Show first 5 dishes
display(df.head())


In [ ]:

# Download the official train/test splits

base = "gs://nutrition5k_dataset/nutrition5k_dataset/dish_ids/splits"

!gsutil cp gs://nutrition5k_dataset/nutrition5k_dataset/dish_ids/splits/rgb_train_ids.txt /content/
!gsutil cp gs://nutrition5k_dataset/nutrition5k_dataset/dish_ids/splits/rgb_test_ids.txt /content/


In [ ]:
import os

for name in ["rgb_train_ids.txt", "rgb_test_ids.txt"]:
    print(name, os.path.exists("/content/" + name))

with open("/content/rgb_train_ids.txt") as f:
    train_ids = set(f.read().splitlines())

with open("/content/rgb_test_ids.txt") as f:
    test_ids = set(f.read().splitlines())

print("Training dishes:", len(train_ids))
print("Testing dishes:", len(test_ids))
print("Overlapping dishes:", len(train_ids & test_ids))

In [ ]:
with open("/content/rgb_train_ids.txt") as f:
    train_ids = set(f.read().splitlines())

with open("/content/rgb_test_ids.txt") as f:
    test_ids = set(f.read().splitlines())

print("Training dishes:", len(train_ids))
print("Testing dishes:", len(test_ids))

print("Overlapping dishes:", len(train_ids & test_ids))

print("Matching training dishes:",
      df["dish_id"].isin(train_ids).sum())

print("Matching testing dishes:",
      df["dish_id"].isin(test_ids).sum())

In [ ]:

import subprocess
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# Folder for sample images
folder = Path("/content/nutrition_samples")
folder.mkdir(exist_ok=True)

base = (
    "gs://nutrition5k_dataset/"
    "nutrition5k_dataset/imagery/realsense_overhead"
)

dish_id = None

# Find a training dish with an available RGB image
candidates = df[df["dish_id"].isin(train_ids)]["dish_id"]

for candidate in candidates.head(30):
    remote = f"{base}/{candidate}/rgb.png"

    check = subprocess.run(
        ["gsutil", "-q", "stat", remote],
        capture_output=True
    )

    if check.returncode == 0:
        dish_id = candidate
        break

if dish_id is None:
    print("No image found in first 30 dishes.")
else:
    image_path = folder / f"{dish_id}.png"

    subprocess.run(
        ["gsutil", "cp", remote, str(image_path)],
        check=True
    )

    # Get nutrition information for this dish
    info = df.set_index("dish_id").loc[dish_id]

    print("Dish ID:", dish_id)
    print("Calories:", round(info["calories_kcal"], 2))
    print("Weight:", info["mass_g"], "g")

    # Display image
    with Image.open(image_path) as image:
        plt.imshow(image.convert("RGB"))

    plt.title(f"{info['calories_kcal']:.1f} kcal")
    plt.axis("off")
    plt.show()


In [ ]:
# Check overhead image dataset size
!gsutil du -sh gs://nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead

# Check available Colab disk space
!df -h /content

In [ ]:
!df -h /content

In [ ]:
!mkdir -p /content/nutrition5k/imagery

!gsutil -m cp -r \
gs://nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead \
/content/nutrition5k/imagery/

In [ ]:

from pathlib import Path

image_root = Path(
    "/content/nutrition5k/imagery/realsense_overhead"
)

# Count downloaded RGB images
rgb_images = list(image_root.glob("*/rgb.png"))

# Count training and testing dishes
available_ids = {p.parent.name for p in rgb_images}

training_count = len(available_ids & train_ids)
testing_count = len(available_ids & test_ids)

print("RGB images:", len(rgb_images))
print("Training images:", training_count)
print("Testing images:", testing_count)
print("Folder exists:", image_root.exists())


In [ ]:

from pathlib import Path

# Main dataset folder
image_root = Path(
    "/content/nutrition5k/imagery/realsense_overhead"
)

# Find available RGB images
rgb_images = list(image_root.glob("*/rgb.png"))

# Connect images to nutrition labels
df["image_path"] = df["dish_id"].apply(
    lambda x: str(image_root / x / "rgb.png")
)

available_df = df[
    df["image_path"].apply(lambda x: Path(x).is_file())
].copy()

# Official train/test split
train_df = available_df[
    available_df["dish_id"].isin(train_ids)
].copy()

test_df = available_df[
    available_df["dish_id"].isin(test_ids)
].copy()

print("RGB images found:", len(rgb_images))
print("Images matched with labels:", len(available_df))
print("Training dishes:", len(train_df))
print("Testing dishes:", len(test_df))
print("Overlap:", len(
    set(train_df["dish_id"]) & set(test_df["dish_id"])
))


In [ ]:

from sklearn.model_selection import train_test_split
import pandas as pd

# Check that calorie labels are valid
clean_df = train_df.copy()

clean_df["calories_kcal"] = pd.to_numeric(
    clean_df["calories_kcal"],
    errors="coerce"
)

clean_df = clean_df.dropna(
    subset=["calories_kcal"]
)

clean_df = clean_df[
    clean_df["calories_kcal"] >= 0
].drop_duplicates("dish_id")

# Split training data into Train and Validation
train_reg_df, val_reg_df = train_test_split(
    clean_df,
    test_size=0.15,
    random_state=42
)

# Check for overlap
assert set(train_reg_df["dish_id"]).isdisjoint(
    set(val_reg_df["dish_id"])
)

print("Training:", len(train_reg_df))
print("Validation:", len(val_reg_df))
print("Testing reserved:", len(test_df))

# Save the split in Google Drive
train_reg_df.to_csv(
    f"{save_dir}/calorie_train.csv", index=False
)

val_reg_df.to_csv(
    f"{save_dir}/calorie_val.csv", index=False
)


In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models import ResNet18_Weights
from PIL import Image

# Preprocessing suitable for pretrained ResNet18
transform = ResNet18_Weights.DEFAULT.transforms()

class CalorieDataset(Dataset):

    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        row = self.data.iloc[index]

        # Read the food image
        with Image.open(row["image_path"]) as img:
            image = transform(img.convert("RGB"))

        # Read its actual calories
        calories = torch.tensor(
            [row["calories_kcal"]],
            dtype=torch.float32
        )

        return image, calories


train_dataset = CalorieDataset(train_reg_df)
val_dataset = CalorieDataset(val_reg_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

images, calories = next(iter(train_loader))

print("Images shape:", images.shape)
print("Calories shape:", calories.shape)
print("Example calories:", calories[0].item())


In [ ]:

import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Load pretrained ResNet18
calorie_model = resnet18(
    weights=ResNet18_Weights.DEFAULT
)

# Freeze the pretrained layers
for param in calorie_model.parameters():
    param.requires_grad = False

# Replace classification layer with regression
calorie_model.fc = nn.Linear(512, 1)

calorie_model = calorie_model.to(device)

print("Device:", device)
print("Output layer:", calorie_model.fc)


In [ ]:

calorie_model.eval()

with torch.no_grad():
    images = images.to(device)
    predictions = calorie_model(images)

print("Predictions shape:", predictions.shape)
print("First raw output:", predictions[0].item())


In [ ]:

import torch.nn as nn

# Statistics from training data ONLY
calorie_mean = train_reg_df["calories_kcal"].mean()
calorie_std = train_reg_df["calories_kcal"].std()

print("Mean calories:", round(calorie_mean, 2))
print("Std calories:", round(calorie_std, 2))

# Simple baseline: always predict the training mean
baseline_mae = (
    val_reg_df["calories_kcal"] - calorie_mean
).abs().mean()

print("Baseline MAE:", round(baseline_mae, 2), "kcal")

# Loss function for regression
criterion = nn.SmoothL1Loss(beta=1.0)

# Train only the new regression layer
optimizer = torch.optim.Adam(
    calorie_model.fc.parameters(),
    lr=0.001
)


In [ ]:

# Keep frozen ResNet18 BatchNorm statistics fixed
calorie_model.eval()

total_loss = 0

# Training
for imgs, targets in train_loader:

    imgs = imgs.to(device)
    targets = targets.to(device)

    # Normalize actual calorie labels
    targets_norm = (targets - calorie_mean) / calorie_std

    # Forward pass
    predictions = calorie_model(imgs)

    # Calculate loss
    loss = criterion(predictions, targets_norm)

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item() * imgs.size(0)

train_loss = total_loss / len(train_dataset)

# Validation: convert predictions back to kcal
calorie_model.eval()
absolute_error = 0

with torch.no_grad():
    for imgs, targets in val_loader:

        imgs = imgs.to(device)
        targets = targets.to(device)

        predictions_norm = calorie_model(imgs)
        predictions_kcal = (
            predictions_norm * calorie_std + calorie_mean
        )

        absolute_error += (
            predictions_kcal - targets
        ).abs().sum().item()

val_mae = absolute_error / len(val_dataset)

print("Epoch 1 Train Loss:", round(train_loss, 4))
print("Epoch 1 Validation MAE:", round(val_mae, 2), "kcal")
print("Baseline MAE:", round(baseline_mae, 2), "kcal")

# Save model and normalization values
torch.save({
    "model_state_dict": calorie_model.state_dict(),
    "calorie_mean": float(calorie_mean),
    "calorie_std": float(calorie_std),
    "epoch": 1,
    "val_mae": val_mae
}, f"{save_dir}/calorie_regression_epoch1.pth")

print("Epoch 1 saved!")


In [ ]:

# Continue training the existing model
calorie_model.eval()

total_loss = 0

for imgs, targets in train_loader:
    imgs = imgs.to(device)
    targets = targets.to(device)

    targets_norm = (targets - calorie_mean) / calorie_std

    predictions = calorie_model(imgs)
    loss = criterion(predictions, targets_norm)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item() * imgs.size(0)

train_loss = total_loss / len(train_dataset)

# Evaluate on Validation
calorie_model.eval()
total_error = 0

with torch.no_grad():
    for imgs, targets in val_loader:
        imgs = imgs.to(device)
        targets = targets.to(device)

        predictions_norm = calorie_model(imgs)
        predictions_kcal = (
            predictions_norm * calorie_std + calorie_mean
        )

        total_error += (
            predictions_kcal - targets
        ).abs().sum().item()

val_mae_2 = total_error / len(val_dataset)

print("Epoch 2 Train Loss:", round(train_loss, 4))
print("Epoch 2 Validation MAE:", round(val_mae_2, 2), "kcal")
print("Epoch 1 Validation MAE: 105.79 kcal")

torch.save({
    "model_state_dict": calorie_model.state_dict(),
    "calorie_mean": float(calorie_mean),
    "calorie_std": float(calorie_std),
    "epoch": 2,
    "val_mae": val_mae_2
}, f"{save_dir}/calorie_regression_epoch2.pth")

print("Epoch 2 saved!")


In [ ]:

# Unfreeze the last ResNet18 block
for param in calorie_model.layer4.parameters():
    param.requires_grad = True

# New optimizer with two learning rates
optimizer = torch.optim.Adam([
    {
        "params": calorie_model.layer4.parameters(),
        "lr": 1e-5
    },
    {
        "params": calorie_model.fc.parameters(),
        "lr": 1e-4
    }
])

print("Fine-Tuning is ready!")


In [ ]:

# Keep BatchNorm statistics fixed
calorie_model.eval()

total_loss = 0

for imgs, targets in train_loader:
    imgs = imgs.to(device)
    targets = targets.to(device)

    targets_norm = (targets - calorie_mean) / calorie_std

    predictions = calorie_model(imgs)
    loss = criterion(predictions, targets_norm)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item() * imgs.size(0)

train_loss_ft = total_loss / len(train_dataset)

# Validation
calorie_model.eval()
total_error = 0

with torch.no_grad():
    for imgs, targets in val_loader:
        imgs = imgs.to(device)
        targets = targets.to(device)

        predictions = calorie_model(imgs)
        predictions_kcal = (
            predictions * calorie_std + calorie_mean
        )

        total_error += (
            predictions_kcal - targets
        ).abs().sum().item()

val_mae_ft = total_error / len(val_dataset)

print("Fine-Tuning Train Loss:", round(train_loss_ft, 4))
print("Fine-Tuning Validation MAE:",
      round(val_mae_ft, 2), "kcal")
print("Previous best MAE: 104.54 kcal")

# Save without overwriting Epoch 2
torch.save({
    "model_state_dict": calorie_model.state_dict(),
    "calorie_mean": float(calorie_mean),
    "calorie_std": float(calorie_std),
    "epoch": 3,
    "val_mae": val_mae_ft
}, f"{save_dir}/calorie_finetune_epoch3.pth")

print("Fine-Tuning checkpoint saved!")


In [ ]:

# Epoch 4: continue fine-tuning
calorie_model.eval()

total_loss = 0

for imgs, targets in train_loader:
    imgs = imgs.to(device)
    targets = targets.to(device)

    targets_norm = (targets - calorie_mean) / calorie_std

    predictions = calorie_model(imgs)
    loss = criterion(predictions, targets_norm)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item() * imgs.size(0)

train_loss_4 = total_loss / len(train_dataset)

# Validation
calorie_model.eval()
total_error = 0

with torch.no_grad():
    for imgs, targets in val_loader:
        imgs = imgs.to(device)
        targets = targets.to(device)

        predictions_kcal = (
            calorie_model(imgs) * calorie_std + calorie_mean
        )

        total_error += (
            predictions_kcal - targets
        ).abs().sum().item()

val_mae_4 = total_error / len(val_dataset)

print("Epoch 4 Train Loss:", round(train_loss_4, 4))
print("Epoch 4 Validation MAE:", round(val_mae_4, 2), "kcal")
print("Previous best:", 89.10, "kcal")

# Keep the best model
if val_mae_4 < val_mae_ft:
    torch.save({
        "model_state_dict": calorie_model.state_dict(),
        "calorie_mean": float(calorie_mean),
        "calorie_std": float(calorie_std),
        "epoch": 4,
        "val_mae": val_mae_4
    }, f"{save_dir}/calorie_best.pth")

    print("New best model saved!")
else:
    print("Epoch 3 remains the best model.")


In [ ]:

# Median of training calories only
median_calories = train_reg_df["calories_kcal"].median()

# Evaluate median baseline on validation
median_baseline_mae = (
    val_reg_df["calories_kcal"] - median_calories
).abs().mean()

print("Training median:", round(median_calories, 2), "kcal")
print("Median Baseline MAE:",
      round(median_baseline_mae, 2), "kcal")
print("Our ResNet18 MAE: 81.01 kcal")


In [ ]:

def train_one_epoch(model, train_loader, val_loader):
    # Freeze BatchNorm running statistics
    model.eval()

    total_loss = 0

    for imgs, targets in train_loader:
        imgs = imgs.to(device)
        targets = targets.to(device)

        targets_norm = (
            targets - calorie_mean
        ) / calorie_std

        predictions = model(imgs)
        loss = criterion(predictions, targets_norm)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)

    train_loss = total_loss / len(train_dataset)

    # Validation
    model.eval()
    total_error = 0

    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs = imgs.to(device)
            targets = targets.to(device)

            predictions_kcal = (
                model(imgs) * calorie_std + calorie_mean
            )

            total_error += (
                predictions_kcal - targets
            ).abs().sum().item()

    val_mae = total_error / len(val_dataset)

    return train_loss, val_mae


# Train Epoch 5
train_loss_5, val_mae_5 = train_one_epoch(
    calorie_model,
    train_loader,
    val_loader
)

print("Epoch 5 Train Loss:", round(train_loss_5, 4))
print("Epoch 5 Validation MAE:", round(val_mae_5, 2), "kcal")

# Save only if better than Epoch 4
if val_mae_5 < val_mae_4:
    torch.save({
        "model_state_dict": calorie_model.state_dict(),
        "calorie_mean": float(calorie_mean),
        "calorie_std": float(calorie_std),
        "epoch": 5,
        "val_mae": val_mae_5
    }, f"{save_dir}/calorie_best.pth")

    print("New best model saved!")
else:
    print("Epoch 4 remains the best.")


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image
import torch

# Evaluation mode
calorie_model.eval()

# Get one validation batch
imgs, targets = next(iter(val_loader))

# Predict calories
with torch.no_grad():
    predictions = calorie_model(imgs.to(device))

    # Convert predictions back to kcal
    predictions = (
        predictions * calorie_std + calorie_mean
    )

predictions = predictions.cpu().squeeze(1).tolist()
targets = targets.squeeze(1).tolist()

# Display 6 food images
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for i, ax in enumerate(axes.flat):

    path = val_dataset.data.iloc[i]["image_path"]

    with Image.open(path) as img:
        ax.imshow(img.convert("RGB"))

    actual = targets[i]
    predicted = predictions[i]
    error = abs(actual - predicted)

    ax.set_title(
        f"Actual: {actual:.1f} kcal\n"
        f"Predicted: {predicted:.1f} kcal\n"
        f"Error: {error:.1f} kcal"
    )

    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
import torch

calorie_model.eval()

actual_list = []
predicted_list = []

with torch.no_grad():
    for imgs, targets in val_loader:

        imgs = imgs.to(device)

        # Predict and convert back to kcal
        predictions = (
            calorie_model(imgs) * calorie_std
            + calorie_mean
        )

        actual_list.extend(
            targets.squeeze(1).tolist()
        )

        predicted_list.extend(
            predictions.cpu().squeeze(1).tolist()
        )

# Create table of predictions
results_df = pd.DataFrame({
    "Actual": actual_list,
    "Predicted": predicted_list
})

results_df["Error"] = (
    results_df["Actual"] - results_df["Predicted"]
).abs()

# Group dishes by actual calories
results_df["Calorie_Range"] = pd.cut(
    results_df["Actual"],
    bins=[0, 100, 300, 600, float("inf")],
    labels=["0-99", "100-299", "300-599", "600+"],
    right=False
)

summary = results_df.groupby(
    "Calorie_Range",
    observed=True
)["Error"].agg(["count", "mean", "median"])

summary.columns = [
    "Number of dishes",
    "MAE (kcal)",
    "Median error (kcal)"
]

print("Overall MAE:",
      round(results_df["Error"].mean(), 2))

display(summary.round(2))


In [ ]:

import torch
from torch.utils.data import DataLoader

# Load best checkpoint
checkpoint = torch.load(
    f"{save_dir}/calorie_best.pth",
    map_location=device,
    weights_only=True
)

calorie_model.load_state_dict(
    checkpoint["model_state_dict"]
)

calorie_mean = checkpoint["calorie_mean"]
calorie_std = checkpoint["calorie_std"]

calorie_model.eval()

# Prepare the official test dataset
test_dataset = CalorieDataset(test_df)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Best Epoch:", checkpoint["epoch"])
print("Saved Validation MAE:", checkpoint["val_mae"])
print("Test dishes:", len(test_dataset))


In [ ]:

actual_parts = []
predicted_parts = []

with torch.no_grad():
    for imgs, targets in test_loader:
        imgs = imgs.to(device)

        # Predict standardized calories
        predictions_norm = calorie_model(imgs)

        # Convert back to kcal
        predictions_kcal = (
            predictions_norm * calorie_std
            + calorie_mean
        )

        actual_parts.append(targets.squeeze(1))
        predicted_parts.append(
            predictions_kcal.squeeze(1).cpu()
        )

actual = torch.cat(actual_parts)
predicted = torch.cat(predicted_parts)

errors = predicted - actual

# Evaluation metrics
test_mae = errors.abs().mean().item()
test_rmse = torch.sqrt((errors ** 2).mean()).item()
test_bias = errors.mean().item()

# Baseline using TRAINING median only
train_median = train_reg_df["calories_kcal"].median()
baseline_test_mae = (
    actual - train_median
).abs().mean().item()

print("=== FINAL TEST RESULTS ===")
print("Test samples:", len(actual))
print("Model MAE:", round(test_mae, 2), "kcal")
print("Model RMSE:", round(test_rmse, 2), "kcal")
print("Mean signed error:", round(test_bias, 2), "kcal")
print("Median Baseline MAE:",
      round(baseline_test_mae, 2), "kcal")


In [ ]:

import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image

# Load saved checkpoint
checkpoint = torch.load(
    f"{save_dir}/calorie_best.pth",
    map_location=device,
    weights_only=True
)

# Rebuild the model architecture
inference_model = resnet18(weights=None)

inference_model.fc = nn.Linear(512, 1)

# Load trained weights
inference_model.load_state_dict(
    checkpoint["model_state_dict"]
)

inference_model = inference_model.to(device)
inference_model.eval()

# Same preprocessing used during training
inference_transform = ResNet18_Weights.DEFAULT.transforms()

# Restore calorie normalization values
cal_mean = checkpoint["calorie_mean"]
cal_std = checkpoint["calorie_std"]


def predict_calories(image_path):

    with Image.open(image_path) as img:
        image = inference_transform(
            img.convert("RGB")
        ).unsqueeze(0).to(device)

    with torch.inference_mode():
        prediction = inference_model(image).item()

    # Convert normalized output to kcal
    predicted_kcal = prediction * cal_std + cal_mean

    return max(0.0, predicted_kcal)


print("Model loaded successfully!")
print("Best epoch:", checkpoint["epoch"])
print("Validation MAE:", checkpoint["val_mae"])


In [ ]:

sample = val_reg_df.iloc[0]

prediction = predict_calories(sample["image_path"])
actual = sample["calories_kcal"]

print("Actual Calories:", round(actual, 2))
print("Predicted Calories:", round(prediction, 2))
print("Absolute Error:", round(abs(actual - prediction), 2))


In [ ]:
import os

save_dir = "/content/drive/MyDrive/NutriVision_AI"

for filename in os.listdir(save_dir):
    if filename.endswith(".pth"):
        path = os.path.join(save_dir, filename)
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"{filename}: {size:.2f} MB")

## More ResNet18 training and evaluation — original cell 60

In [ ]:
import torch

checkpoint_path = f"{save_dir}/resnet18_fullfinetune_epoch3.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint keys:", list(checkpoint.keys()))

for key in [
    "epoch",
    "val_accuracy",
    "val_acc",
    "train_accuracy",
    "train_acc",
    "best_val_acc"
]:
    if key in checkpoint:
        print(f"{key}: {checkpoint[key]}")

print(
    "Has optimizer state:",
    "optimizer_state_dict" in checkpoint
)

print(
    "Has model weights:",
    "model_state_dict" in checkpoint
)

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_dir = "/content/drive/MyDrive/NutriVision_AI"

# Restore original Food-101 split
split = torch.load(
    f"{save_dir}/data_split.pth",
    weights_only=True
)

# Use the same preprocessing as before
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

val_transform = ResNet18_Weights.DEFAULT.transforms()

food_train_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=train_transform,
    download=True
)

food_val_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=val_transform,
    download=False
)

food_train_subset = Subset(
    food_train_data, split["train_indices"]
)

food_val_subset = Subset(
    food_val_data, split["val_indices"]
)

food_train_loader = DataLoader(
    food_train_subset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

food_val_loader = DataLoader(
    food_val_subset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# Recreate the classification model
food_model = resnet18(weights=None)
food_model.fc = nn.Linear(512, 101)

# Load the trained weights from epoch 3
checkpoint = torch.load(
    f"{save_dir}/resnet18_fullfinetune_epoch3.pth",
    map_location="cpu",
    weights_only=True
)

food_model.load_state_dict(
    checkpoint["model_state_dict"]
)

food_model = food_model.to(device)

print("Model restored successfully!")
print("Previous epoch:", checkpoint["epoch"])
print("Previous Val Accuracy:", checkpoint["val_accuracy"])
print("Training images:", len(food_train_subset))
print("Validation images:", len(food_val_subset))
print("Device:", device)

In [ ]:

import torch
import torch.nn as nn
from tqdm.auto import tqdm

# 1. Enable all ResNet18 layers to learn
for param in food_model.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()

# 2. Different learning rates for backbone and classifier
backbone_params = [
    param for name, param in food_model.named_parameters()
    if not name.startswith("fc.")
]

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 1e-5},
        {"params": food_model.fc.parameters(), "lr": 1e-4}
    ],
    weight_decay=1e-4
)

# 3. Gradually decrease learning rates
epochs = 5

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs
)

# 4. GPU acceleration using mixed precision
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

if use_amp:
    torch.backends.cudnn.benchmark = True

# 5. Keep the best previous accuracy
best_val_acc = checkpoint["val_accuracy"]
start_epoch = checkpoint["epoch"]

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

def run_food_epoch(loader, training):

    if training:
        food_model.train()
    else:
        food_model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in tqdm(loader, leave=False):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):

            with torch.autocast(
                device_type=device.type,
                enabled=use_amp
            ):
                outputs = food_model(images)
                loss = criterion(outputs, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()

        total_samples += batch_size

    return (
        total_loss / total_samples,
        100 * total_correct / total_samples
    )


# 6. Continue training from epoch 3
for i in range(epochs):

    epoch_number = start_epoch + i + 1

    train_loss, train_acc = run_food_epoch(
        food_train_loader,
        training=True
    )

    val_loss, val_acc = run_food_epoch(
        food_val_loader,
        training=False
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    scheduler.step()

    print(f"\nEpoch {epoch_number}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save({
            "epoch": epoch_number,
            "model_state_dict": food_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_accuracy": val_acc,
            "val_loss": val_loss,
            "train_accuracy": train_acc,
            "classes": food_train_data.classes
        }, f"{save_dir}/resnet18_improved_best.pth")

        print("NEW BEST MODEL SAVED!")

    print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

print("\nTraining completed!")
print(f"FINAL BEST: {best_val_acc:.2f}%")


## ConvNeXt Food-101 training — original cell 63

In [ ]:
%pip -q install -U timm

In [ ]:

import torch
import timm

from torchvision import datasets, transforms as T
from torch.utils.data import DataLoader, Subset

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

save_dir = "/content/drive/MyDrive/NutriVision_AI"

# Keep the original Food-101 split
split = torch.load(
    f"{save_dir}/data_split.pth",
    weights_only=True
)

train_ids = split["train_indices"]
val_ids = split["val_indices"]

assert set(map(int, train_ids)).isdisjoint(
    set(map(int, val_ids))
), "Data leakage detected!"

# Strong pretrained classification backbone
model_name = "convnext_base.fb_in22k_ft_in1k_384"

fusion_cnn = timm.create_model(
    model_name,
    pretrained=True,
    num_classes=101
)

# Read the model's correct preprocessing settings
data_cfg = timm.data.resolve_model_data_config(
    fusion_cnn
)

mean = data_cfg["mean"]
std = data_cfg["std"]

# Moderate augmentation for food photographs
fusion_train_transform = T.Compose([
    T.RandomResizedCrop(
        384,
        scale=(0.65, 1.0),
        interpolation=T.InterpolationMode.BICUBIC
    ),
    T.RandomHorizontalFlip(),
    T.RandAugment(num_ops=2, magnitude=7),
    T.ToTensor(),
    T.Normalize(mean, std)
])

# Deterministic validation preprocessing
fusion_val_transform = timm.data.create_transform(
    **data_cfg,
    is_training=False
)

train_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=fusion_train_transform,
    download=True
)

val_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=fusion_val_transform,
    download=False
)

assert train_data.classes == val_data.classes

fusion_train = Subset(train_data, train_ids)
fusion_val = Subset(val_data, val_ids)

fusion_train_loader = DataLoader(
    fusion_train,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

fusion_val_loader = DataLoader(
    fusion_val,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

fusion_cnn = fusion_cnn.to(device)

# Sanity check before expensive training
images, labels = next(iter(fusion_val_loader))
images = images.to(device)

fusion_cnn.eval()

with torch.inference_mode():
    with torch.autocast(
        device_type="cuda",
        enabled=(device.type == "cuda")
    ):
        outputs = fusion_cnn(images)

print("Model:", model_name)
print("Parameters:", sum(
    p.numel() for p in fusion_cnn.parameters()
) / 1e6, "million")
print("Images:", images.shape)
print("Outputs:", outputs.shape)
print("Training images:", len(fusion_train))
print("Validation images:", len(fusion_val))
print("Device:", device)


In [ ]:

import torch
import torch.nn as nn
from tqdm.auto import tqdm

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Freeze the pretrained backbone
for param in fusion_cnn.parameters():
    param.requires_grad = False

# Train only the classification head
for param in fusion_cnn.head.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    fusion_cnn.head.parameters(),
    lr=5e-4,
    weight_decay=0.01
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_acc = -1.0

for epoch in range(1, 3):

    # Keep the frozen backbone in evaluation mode
    fusion_cnn.eval()
    fusion_cnn.head.train()

    train_correct = 0
    train_total = 0
    train_loss_sum = 0.0

    for images, labels in tqdm(
        fusion_train_loader,
        desc=f"Training Epoch {epoch}"
    ):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            enabled=use_amp
        ):
            outputs = fusion_cnn(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)
        train_loss_sum += loss.item() * batch_size
        train_correct += (
            outputs.argmax(1) == labels
        ).sum().item()
        train_total += batch_size

    # Validation
    fusion_cnn.eval()

    val_correct = 0
    val_total = 0
    val_loss_sum = 0.0

    with torch.inference_mode():
        for images, labels in tqdm(
            fusion_val_loader,
            desc=f"Validation Epoch {epoch}"
        ):

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                enabled=use_amp
            ):
                outputs = fusion_cnn(images)
                loss = criterion(outputs, labels)

            batch_size = labels.size(0)
            val_loss_sum += loss.item() * batch_size
            val_correct += (
                outputs.argmax(1) == labels
            ).sum().item()
            val_total += batch_size

    train_acc = 100 * train_correct / train_total
    val_acc = 100 * val_correct / val_total
    train_loss = train_loss_sum / train_total
    val_loss = val_loss_sum / val_total

    print(f"\nEpoch {epoch}/2")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc

        torch.save({
            "epoch": epoch,
            "model_state_dict": fusion_cnn.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_accuracy": val_acc,
            "classes": train_data.classes,
            "model_name": model_name
        }, f"{save_dir}/convnext_head_best.pth")

        print("Best checkpoint saved!")

print(f"\nBest Head Accuracy: {best_acc:.2f}%")


In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# 1. Restore the best head-training checkpoint
head_checkpoint = torch.load(
    f"{save_dir}/convnext_head_best.pth",
    map_location="cpu",
    weights_only=True
)

fusion_cnn.load_state_dict(
    head_checkpoint["model_state_dict"]
)
fusion_cnn = fusion_cnn.to(device)

print("Restored accuracy:", head_checkpoint["val_accuracy"])

# 2. Freeze all parameters
for param in fusion_cnn.parameters():
    param.requires_grad = False

# Unfreeze the last ConvNeXt stage
for param in fusion_cnn.stages[3].parameters():
    param.requires_grad = True

# Unfreeze the classification head
for param in fusion_cnn.head.parameters():
    param.requires_grad = True

# 3. Smaller batch size for GPU memory safety
fine_train_loader = DataLoader(
    fusion_train,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

fine_val_loader = DataLoader(
    fusion_val,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# 4. Different learning rates
optimizer = torch.optim.AdamW(
    [
        {
            "params": fusion_cnn.stages[3].parameters(),
            "lr": 1e-5
        },
        {
            "params": fusion_cnn.head.parameters(),
            "lr": 5e-5
        }
    ],
    weight_decay=0.01
)

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

epochs = 3
accumulation_steps = 2

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-6
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_val_acc = head_checkpoint["val_accuracy"]

# 5. Train and validate
for epoch in range(1, epochs + 1):

    fusion_cnn.train()
    optimizer.zero_grad(set_to_none=True)

    train_correct = 0
    train_total = 0
    train_loss_sum = 0.0

    for step, (images, labels) in enumerate(
        tqdm(fine_train_loader, desc=f"Train {epoch}")
    ):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(
            device_type=device.type,
            enabled=use_amp
        ):
            outputs = fusion_cnn(images)
            loss = criterion(outputs, labels)

        # Gradient accumulation
        scaler.scale(
            loss / accumulation_steps
        ).backward()

        update_now = (
            (step + 1) % accumulation_steps == 0
            or (step + 1) == len(fine_train_loader)
        )

        if update_now:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                filter(
                    lambda p: p.requires_grad,
                    fusion_cnn.parameters()
                ),
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        batch_size = labels.size(0)

        train_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()

        train_total += batch_size
        train_loss_sum += loss.item() * batch_size

    # Validation
    fusion_cnn.eval()

    val_correct = 0
    val_total = 0
    val_loss_sum = 0.0

    with torch.inference_mode():
        for images, labels in tqdm(
            fine_val_loader,
            desc=f"Validation {epoch}"
        ):

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type=device.type,
                enabled=use_amp
            ):
                outputs = fusion_cnn(images)
                loss = criterion(outputs, labels)

            batch_size = labels.size(0)

            val_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()

            val_total += batch_size
            val_loss_sum += loss.item() * batch_size

    train_acc = 100 * train_correct / train_total
    val_acc = 100 * val_correct / val_total

    train_loss = train_loss_sum / train_total
    val_loss = val_loss_sum / val_total

    scheduler.step()

    print(f"\nEpoch {epoch}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save({
            "epoch": epoch,
            "model_state_dict": fusion_cnn.state_dict(),
            "val_accuracy": val_acc,
            "train_accuracy": train_acc,
            "model_name": model_name,
            "classes": train_data.classes
        }, f"{save_dir}/convnext_stage3_best.pth")

        print("NEW BEST MODEL SAVED!")

    print(f"Best Accuracy: {best_val_acc:.2f}%")

print("\nFine-tuning finished!")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")


In [ ]:

import torch
from collections import Counter
from tqdm.auto import tqdm

# Restore the best fine-tuned ConvNeXt
best_checkpoint = torch.load(
    f"{save_dir}/convnext_stage3_best.pth",
    map_location="cpu",
    weights_only=True
)

fusion_cnn.load_state_dict(
    best_checkpoint["model_state_dict"]
)

fusion_cnn = fusion_cnn.to(device)
fusion_cnn.eval()

class_names = best_checkpoint["classes"]

original_correct = 0
tta_correct = 0
top5_correct = 0
total = 0

class_total = Counter()
class_correct = Counter()
confusions = Counter()

with torch.inference_mode():

    for images, labels in tqdm(
        fine_val_loader,
        desc="TTA Evaluation"
    ):

        images = images.to(device)
        labels = labels.to(device)

        with torch.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda")
        ):

            # Original image
            outputs_original = fusion_cnn(images)

            # Horizontally flipped image
            flipped_images = torch.flip(
                images, dims=[3]
            )

            outputs_flipped = fusion_cnn(
                flipped_images
            )

        # Convert logits into probabilities
        probs_original = torch.softmax(
            outputs_original.float(), dim=1
        )

        probs_flipped = torch.softmax(
            outputs_flipped.float(), dim=1
        )

        # Average the two predictions
        tta_probs = (
            probs_original + probs_flipped
        ) / 2

        original_preds = outputs_original.argmax(1)
        tta_preds = tta_probs.argmax(1)

        top5_preds = tta_probs.topk(
            5, dim=1
        ).indices

        original_correct += (
            original_preds == labels
        ).sum().item()

        tta_correct += (
            tta_preds == labels
        ).sum().item()

        top5_correct += (
            top5_preds == labels.unsqueeze(1)
        ).any(dim=1).sum().item()

        total += labels.size(0)

        for actual, predicted in zip(
            labels.cpu().tolist(),
            tta_preds.cpu().tolist()
        ):
            class_total[actual] += 1

            if actual == predicted:
                class_correct[actual] += 1
            else:
                confusions[(actual, predicted)] += 1

original_acc = 100 * original_correct / total
tta_acc = 100 * tta_correct / total
top5_acc = 100 * top5_correct / total

print("\n===== NUTRIVISION TTA RESULTS =====")
print(f"Original Top-1: {original_acc:.2f}%")
print(f"TTA Top-1: {tta_acc:.2f}%")
print(f"TTA Top-5: {top5_acc:.2f}%")
print(f"Improvement: {tta_acc-original_acc:+.2f} pp")

print("\n===== HARDEST 10 CLASSES =====")

class_scores = [
    (
        100 * class_correct[i] / class_total[i],
        class_names[i]
    )
    for i in range(len(class_names))
    if class_total[i] > 0
]

for acc, name in sorted(class_scores)[:10]:
    print(f"{name}: {acc:.2f}%")

print("\n===== TOP CONFUSIONS =====")

for (actual, predicted), count in confusions.most_common(10):
    print(
        f"{class_names[actual]} -> "
        f"{class_names[predicted]}: {count}"
    )


## DINOv2 head training — original cell 68

In [ ]:

import gc
import torch
import timm

# Free GPU memory from ConvNeXt
if "fusion_cnn" in globals():
    del fusion_cnn

if "outputs" in globals():
    del outputs

gc.collect()
torch.cuda.empty_cache()

# Reproducible classifier initialization
torch.manual_seed(42)

dino_name = "vit_base_patch14_dinov2.lvd142m"

# Load pretrained DINOv2 with a new Food-101 classifier
dino_model = timm.create_model(
    dino_name,
    pretrained=True,
    num_classes=101,
    img_size=280
).to(device)

dino_model.eval()

# Test a small batch before training
test_images = torch.randn(
    2, 3, 280, 280,
    device=device
)

with torch.inference_mode():
    with torch.autocast(
        device_type="cuda",
        enabled=(device.type == "cuda")
    ):
        test_outputs = dino_model(test_images)

print("Model:", dino_name)
print("Input:", test_images.shape)
print("Output:", test_outputs.shape)
print("Parameters:", round(
    sum(p.numel() for p in dino_model.parameters()) / 1e6, 2
), "million")

if device.type == "cuda":
    print(
        "GPU allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

assert test_outputs.shape == (2, 101)
print("DINOv2 ready!")


In [ ]:

import torch
import timm
from torchvision import datasets, transforms as T
from torch.utils.data import DataLoader, Subset

# Use DINOv2 pretrained normalization
dino_cfg = timm.data.resolve_model_data_config(dino_model)

mean = dino_cfg["mean"]
std = dino_cfg["std"]

dino_train_transform = T.Compose([
    T.RandomResizedCrop(
        280,
        scale=(0.7, 1.0),
        interpolation=T.InterpolationMode.BICUBIC
    ),
    T.RandomHorizontalFlip(),
    T.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1
    ),
    T.ToTensor(),
    T.Normalize(mean, std)
])

dino_val_transform = T.Compose([
    T.Resize(
        320,
        interpolation=T.InterpolationMode.BICUBIC
    ),
    T.CenterCrop(280),
    T.ToTensor(),
    T.Normalize(mean, std)
])

dino_train_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=dino_train_transform,
    download=False
)

dino_val_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=dino_val_transform,
    download=False
)

# Reuse exactly the same train/validation split
dino_train = Subset(
    dino_train_data,
    split["train_indices"]
)

dino_val = Subset(
    dino_val_data,
    split["val_indices"]
)

dino_train_loader = DataLoader(
    dino_train,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

dino_val_loader = DataLoader(
    dino_val,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Training images:", len(dino_train))
print("Validation images:", len(dino_val))
print("Number of classes:", len(dino_train_data.classes))
print("DINOv2 data ready!")


In [ ]:

import torch
import torch.nn as nn
from tqdm.auto import tqdm

# Freeze all pretrained layers
for param in dino_model.parameters():
    param.requires_grad = False

# Allow only the classifier to learn
for param in dino_model.head.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = torch.optim.AdamW(
    dino_model.head.parameters(),
    lr=5e-4,
    weight_decay=0.01
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

epochs = 3
best_dino_acc = -1.0

for epoch in range(1, epochs + 1):

    # Frozen backbone remains in evaluation mode
    dino_model.eval()
    dino_model.head.train()

    train_correct = 0
    train_total = 0
    train_loss_sum = 0.0

    for images, labels in tqdm(
        dino_train_loader,
        desc=f"DINO Train {epoch}"
    ):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            enabled=use_amp
        ):
            outputs = dino_model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = labels.size(0)

        train_loss_sum += loss.item() * bs
        train_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()
        train_total += bs

    # Validation
    dino_model.eval()

    val_correct = 0
    val_total = 0
    val_loss_sum = 0.0

    with torch.inference_mode():
        for images, labels in tqdm(
            dino_val_loader,
            desc=f"DINO Validation {epoch}"
        ):

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type=device.type,
                enabled=use_amp
            ):
                outputs = dino_model(images)
                loss = criterion(outputs, labels)

            bs = labels.size(0)

            val_loss_sum += loss.item() * bs
            val_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            val_total += bs

    train_acc = 100 * train_correct / train_total
    val_acc = 100 * val_correct / val_total

    print(f"\n===== DINOv2 Epoch {epoch}/{epochs} =====")
    print(f"Train Loss: {train_loss_sum/train_total:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Loss: {val_loss_sum/val_total:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    if val_acc > best_dino_acc:

        best_dino_acc = val_acc

        torch.save({
            "epoch": epoch,
            "model_state_dict": dino_model.state_dict(),
            "val_accuracy": val_acc,
            "classes": dino_train_data.classes,
            "model_name": dino_name,
            "image_size": 280
        }, f"{save_dir}/dinov2_head_best.pth")

        print("NEW BEST DINOv2 SAVED!")

    print(f"Best DINOv2 Accuracy: {best_dino_acc:.2f}%")

print("\nDINOv2 head training completed!")
print(f"Best Accuracy: {best_dino_acc:.2f}%")


## Classification ensemble evaluation / DINOv2 fine-tuning — original cell 71

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/NutriVision_AI"

print("Folder exists:", os.path.isdir(save_dir))
print("ConvNeXt checkpoint:", os.path.exists(
    f"{save_dir}/convnext_stage3_best.pth"
))
print("DINOv2 checkpoint:", os.path.exists(
    f"{save_dir}/dinov2_head_best.pth"
))

In [ ]:

import torch
import timm
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Load the best ConvNeXt
conv_ckpt = torch.load(
    f"{save_dir}/convnext_stage3_best.pth",
    map_location="cpu",
    weights_only=True
)

conv_model = timm.create_model(
    "convnext_base.fb_in22k_ft_in1k_384",
    pretrained=False,
    num_classes=101
)

conv_model.load_state_dict(
    conv_ckpt["model_state_dict"]
)
conv_model = conv_model.to(device).eval()

# Restore the best DINOv2
dino_ckpt = torch.load(
    f"{save_dir}/dinov2_head_best.pth",
    map_location="cpu",
    weights_only=True
)

dino_model.load_state_dict(
    dino_ckpt["model_state_dict"]
)
dino_model = dino_model.to(device).eval()

# Ensure identical class order and validation images
assert conv_ckpt["classes"] == dino_ckpt["classes"]
assert list(map(int, fusion_val.indices)) == list(
    map(int, dino_val.indices)
)

# Same images, different preprocessing for each model
conv_loader = DataLoader(
    fusion_val,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

dino_loader = DataLoader(
    dino_val,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

conv_probs = []
dino_probs = []
all_labels = []

with torch.inference_mode():

    for (xc, yc), (xd, yd) in tqdm(
        zip(conv_loader, dino_loader),
        total=len(conv_loader),
        desc="NutriVision Fusion"
    ):

        assert torch.equal(yc, yd)

        xc = xc.to(device)
        xd = xd.to(device)

        with torch.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda")
        ):
            out_conv = conv_model(xc)
            out_dino = dino_model(xd)

        conv_probs.append(
            torch.softmax(out_conv.float(), dim=1).cpu()
        )

        dino_probs.append(
            torch.softmax(out_dino.float(), dim=1).cpu()
        )

        all_labels.append(yc)

conv_probs = torch.cat(conv_probs)
dino_probs = torch.cat(dino_probs)
labels = torch.cat(all_labels)

def accuracy(probs):
    return 100 * (
        probs.argmax(dim=1) == labels
    ).float().mean().item()

print("\n===== INDIVIDUAL MODELS =====")
print(f"ConvNeXt: {accuracy(conv_probs):.2f}%")
print(f"DINOv2:   {accuracy(dino_probs):.2f}%")

print("\n===== ENSEMBLE EXPERIMENT =====")

best_weight = None
best_accuracy = -1

for weight in [0.25, 0.50, 0.75]:

    fused = (
        weight * conv_probs
        + (1 - weight) * dino_probs
    )

    acc = accuracy(fused)

    print(
        f"ConvNeXt {weight:.0%} + "
        f"DINOv2 {1-weight:.0%}: {acc:.2f}%"
    )

    if acc > best_accuracy:
        best_accuracy = acc
        best_weight = weight

# Check how many different errors the models make
conv_correct = conv_probs.argmax(1) == labels
dino_correct = dino_probs.argmax(1) == labels

print("\n===== ERROR COMPLEMENTARITY =====")
print("Both correct:", (conv_correct & dino_correct).sum().item())
print("Only ConvNeXt correct:", (conv_correct & ~dino_correct).sum().item())
print("Only DINOv2 correct:", (~conv_correct & dino_correct).sum().item())
print("Both wrong:", (~conv_correct & ~dino_correct).sum().item())

print("\nBest ensemble accuracy:", round(best_accuracy, 2))
print("Best ConvNeXt weight:", best_weight)

# Save validation predictions for later experiments
torch.save({
    "conv_probs": conv_probs,
    "dino_probs": dino_probs,
    "labels": labels,
    "classes": conv_ckpt["classes"]
}, f"{save_dir}/fusion_validation_predictions.pth")


In [ ]:

import torch
import timm
import gc

from torchvision import datasets, transforms as T
from torch.utils.data import Subset

# 1. Restore environment
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

save_dir = "/content/drive/MyDrive/NutriVision_AI"

split = torch.load(
    f"{save_dir}/data_split.pth",
    weights_only=True
)

val_indices = split["val_indices"]

# 2. Restore DINOv2 architecture
dino_model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    num_classes=101,
    img_size=280
)

dino_cfg = timm.data.resolve_model_data_config(
    dino_model
)

# 3. Restore ConvNeXt validation preprocessing
conv_temp = timm.create_model(
    "convnext_base.fb_in22k_ft_in1k_384",
    pretrained=False,
    num_classes=101
)

conv_cfg = timm.data.resolve_model_data_config(
    conv_temp
)

conv_val_transform = timm.data.create_transform(
    **conv_cfg,
    is_training=False
)

del conv_temp
gc.collect()

# 4. Restore DINOv2 validation preprocessing
dino_val_transform = T.Compose([
    T.Resize(
        320,
        interpolation=T.InterpolationMode.BICUBIC
    ),
    T.CenterCrop(280),
    T.ToTensor(),
    T.Normalize(
        dino_cfg["mean"],
        dino_cfg["std"]
    )
])

# 5. Restore identical validation images
conv_val_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=conv_val_transform,
    download=False
)

dino_val_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=dino_val_transform,
    download=False
)

fusion_val = Subset(
    conv_val_data,
    val_indices
)

dino_val = Subset(
    dino_val_data,
    val_indices
)

assert conv_val_data.classes == dino_val_data.classes

print("Device:", device)
print("ConvNeXt validation:", len(fusion_val))
print("DINOv2 validation:", len(dino_val))
print("All variables restored successfully!")


In [ ]:

import os
import torch
import timm

from google.colab import drive
from torchvision import datasets, transforms as T
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# ======================================
# 1. RESTORE ENVIRONMENT
# ======================================

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

save_dir = "/content/drive/MyDrive/NutriVision_AI"

split = torch.load(
    f"{save_dir}/data_split.pth",
    map_location="cpu",
    weights_only=True
)

print("Device:", device)

# ======================================
# 2. LOAD CONVNEXT CHECKPOINT
# ======================================

conv_ckpt = torch.load(
    f"{save_dir}/convnext_stage3_best.pth",
    map_location="cpu",
    weights_only=True
)

conv_model = timm.create_model(
    "convnext_base.fb_in22k_ft_in1k_384",
    pretrained=False,
    num_classes=101
)

conv_model.load_state_dict(
    conv_ckpt["model_state_dict"]
)

conv_cfg = timm.data.resolve_model_data_config(
    conv_model
)

conv_transform = timm.data.create_transform(
    **conv_cfg,
    is_training=False
)

conv_model = conv_model.to(device).eval()

print("ConvNeXt loaded!")

# ======================================
# 3. LOAD DINOV2 CHECKPOINT
# ======================================

dino_ckpt = torch.load(
    f"{save_dir}/dinov2_head_best.pth",
    map_location="cpu",
    weights_only=True
)

dino_model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    num_classes=101,
    img_size=280
)

dino_model.load_state_dict(
    dino_ckpt["model_state_dict"]
)

dino_cfg = timm.data.resolve_model_data_config(
    dino_model
)

dino_transform = T.Compose([
    T.Resize(
        320,
        interpolation=T.InterpolationMode.BICUBIC
    ),
    T.CenterCrop(280),
    T.ToTensor(),
    T.Normalize(
        dino_cfg["mean"],
        dino_cfg["std"]
    )
])

dino_model = dino_model.to(device).eval()

print("DINOv2 loaded!")

# ======================================
# 4. RESTORE FOOD-101 DATASET
# ======================================

print("Checking Food-101 dataset...")

conv_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=conv_transform,
    download=True
)

dino_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=dino_transform,
    download=False
)

assert conv_data.classes == dino_data.classes
assert list(conv_ckpt["classes"]) == list(dino_ckpt["classes"])
assert list(conv_ckpt["classes"]) == list(conv_data.classes)

val_indices = split["val_indices"]

conv_val = Subset(conv_data, val_indices)
dino_val = Subset(dino_data, val_indices)

conv_loader = DataLoader(
    conv_val,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

dino_loader = DataLoader(
    dino_val,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

print("Validation images:", len(conv_val))
print("Dataset ready!")

# ======================================
# 5. RUN BOTH MODELS
# ======================================

conv_probs = []
dino_probs = []
all_labels = []

with torch.inference_mode():

    for (xc, yc), (xd, yd) in tqdm(
        zip(conv_loader, dino_loader),
        total=len(conv_loader),
        desc="NutriVision Fusion"
    ):

        assert torch.equal(yc, yd)

        xc = xc.to(device)
        xd = xd.to(device)

        with torch.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda")
        ):
            out_conv = conv_model(xc)
            out_dino = dino_model(xd)

        conv_probs.append(
            torch.softmax(out_conv.float(), dim=1).cpu()
        )

        dino_probs.append(
            torch.softmax(out_dino.float(), dim=1).cpu()
        )

        all_labels.append(yc)

conv_probs = torch.cat(conv_probs)
dino_probs = torch.cat(dino_probs)
labels = torch.cat(all_labels)

# ======================================
# 6. COMPARE ACCURACY
# ======================================

def accuracy(probs):
    return 100 * (
        probs.argmax(1) == labels
    ).float().mean().item()

print("\n===== INDIVIDUAL MODELS =====")
print(f"ConvNeXt: {accuracy(conv_probs):.2f}%")
print(f"DINOv2: {accuracy(dino_probs):.2f}%")

print("\n===== ENSEMBLE RESULTS =====")

best_accuracy = -1
best_weight = None

for weight in [0.25, 0.50, 0.75]:

    fused = (
        weight * conv_probs
        + (1 - weight) * dino_probs
    )

    acc = accuracy(fused)

    print(
        f"ConvNeXt {weight:.0%} + "
        f"DINOv2 {1-weight:.0%}: {acc:.2f}%"
    )

    if acc > best_accuracy:
        best_accuracy = acc
        best_weight = weight

# ======================================
# 7. ANALYZE ERRORS
# ======================================

conv_correct = conv_probs.argmax(1) == labels
dino_correct = dino_probs.argmax(1) == labels

print("\n===== ERROR ANALYSIS =====")
print("Both correct:",
      (conv_correct & dino_correct).sum().item())

print("Only ConvNeXt correct:",
      (conv_correct & ~dino_correct).sum().item())

print("Only DINOv2 correct:",
      (~conv_correct & dino_correct).sum().item())

print("Both wrong:",
      (~conv_correct & ~dino_correct).sum().item())

print("\nBEST ENSEMBLE:", round(best_accuracy, 2), "%")
print("Best ConvNeXt weight:", best_weight)

# ======================================
# 8. SAVE PREDICTIONS
# ======================================

torch.save({
    "conv_probs": conv_probs,
    "dino_probs": dino_probs,
    "labels": labels,
    "classes": conv_data.classes,
    "best_weight": best_weight
}, f"{save_dir}/fusion_validation_predictions.pth")

print("\nResults saved successfully!")


In [ ]:

import gc
import torch
import torch.nn as nn

from torchvision import datasets, transforms as T
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# Free GPU memory occupied by ConvNeXt
if "conv_model" in globals():
    conv_model.cpu()
    del conv_model

gc.collect()
torch.cuda.empty_cache()

# Restore the best DINOv2 head checkpoint
head_ckpt = torch.load(
    f"{save_dir}/dinov2_head_best.pth",
    map_location="cpu",
    weights_only=True
)

dino_model.load_state_dict(head_ckpt["model_state_dict"])
dino_model = dino_model.to(device)

# Freeze pretrained parameters
for p in dino_model.parameters():
    p.requires_grad = False

# Unfreeze final two transformer blocks
for p in dino_model.blocks[-2:].parameters():
    p.requires_grad = True

# Unfreeze final normalization and classifier
for p in dino_model.norm.parameters():
    p.requires_grad = True

for p in dino_model.head.parameters():
    p.requires_grad = True

print("Trainable parameters:", sum(
    p.numel() for p in dino_model.parameters()
    if p.requires_grad
))

# Restore training augmentation
dino_cfg = timm.data.resolve_model_data_config(dino_model)

train_transform = T.Compose([
    T.RandomResizedCrop(
        280,
        scale=(0.7, 1.0),
        interpolation=T.InterpolationMode.BICUBIC
    ),
    T.RandomHorizontalFlip(),
    T.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1
    ),
    T.ToTensor(),
    T.Normalize(
        dino_cfg["mean"],
        dino_cfg["std"]
    )
])

train_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=train_transform,
    download=False
)

train_set = Subset(train_data, split["train_indices"])

train_loader = DataLoader(
    train_set,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    dino_val,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# Optimizer with differential learning rates
optimizer = torch.optim.AdamW([
    {
        "params": dino_model.blocks[-2:].parameters(),
        "lr": 1e-5
    },
    {
        "params": dino_model.norm.parameters(),
        "lr": 1e-5
    },
    {
        "params": dino_model.head.parameters(),
        "lr": 5e-5
    }
], weight_decay=0.01)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

epochs = 3
accumulation_steps = 2

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-6
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_accuracy = head_ckpt["val_accuracy"]

for epoch in range(1, epochs + 1):

    dino_model.train()
    optimizer.zero_grad(set_to_none=True)

    train_correct = 0
    train_total = 0
    train_loss_sum = 0

    for step, (images, labels) in enumerate(
        tqdm(train_loader, desc=f"Fine-tune {epoch}")
    ):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(
            device_type=device.type,
            enabled=use_amp
        ):
            outputs = dino_model(images)
            loss = criterion(outputs, labels)

        scaler.scale(
            loss / accumulation_steps
        ).backward()

        if (
            (step + 1) % accumulation_steps == 0
            or (step + 1) == len(train_loader)
        ):
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                filter(
                    lambda p: p.requires_grad,
                    dino_model.parameters()
                ),
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        bs = labels.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += bs
        train_loss_sum += loss.item() * bs

    # Validation
    dino_model.eval()

    val_correct = 0
    val_total = 0
    val_loss_sum = 0

    with torch.inference_mode():
        for images, labels in tqdm(
            val_loader,
            desc=f"Validate {epoch}"
        ):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type=device.type,
                enabled=use_amp
            ):
                outputs = dino_model(images)
                loss = criterion(outputs, labels)

            bs = labels.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += bs
            val_loss_sum += loss.item() * bs

    train_acc = 100 * train_correct / train_total
    val_acc = 100 * val_correct / val_total
    val_loss = val_loss_sum / val_total

    scheduler.step()

    print(f"\n===== DINOv2 Fine-tuning {epoch}/{epochs} =====")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Accuracy: {val_acc:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")

    if val_acc > best_accuracy:
        best_accuracy = val_acc

        torch.save({
            "epoch": epoch,
            "model_state_dict": dino_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_accuracy": val_acc,
            "val_loss": val_loss,
            "classes": train_data.classes,
            "model_name": "vit_base_patch14_dinov2.lvd142m",
            "image_size": 280
        }, f"{save_dir}/dinov2_last2_best.pth")

        print("NEW BEST DINOv2 SAVED!")

    print(f"Best DINOv2 Accuracy: {best_accuracy:.2f}%")

print("\nFine-tuning completed!")
print(f"Best DINOv2 Accuracy: {best_accuracy:.2f}%")


In [ ]:

import os
import torch
import torch.nn.functional as F
import timm

from google.colab import drive
from torchvision import datasets, transforms as T
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# 1. Restore environment
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/NutriVision_AI"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load previously saved ConvNeXt predictions
old = torch.load(
    f"{save_dir}/fusion_validation_predictions.pth",
    map_location="cpu",
    weights_only=True
)

conv_probs = old["conv_probs"].float()
labels = old["labels"].long()

# 3. Load improved DINOv2 checkpoint
ckpt = torch.load(
    f"{save_dir}/dinov2_last2_best.pth",
    map_location="cpu",
    weights_only=True
)

dino_model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    num_classes=101,
    img_size=280
)

dino_model.load_state_dict(ckpt["model_state_dict"])
dino_model = dino_model.to(device).eval()

# 4. Same validation preprocessing used during training
cfg = timm.data.resolve_model_data_config(dino_model)

val_transform = T.Compose([
    T.Resize(320, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(280),
    T.ToTensor(),
    T.Normalize(cfg["mean"], cfg["std"])
])

data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=val_transform,
    download=True
)

split = torch.load(
    f"{save_dir}/data_split.pth",
    map_location="cpu",
    weights_only=True
)

assert list(old["classes"]) == list(ckpt["classes"])
assert list(old["classes"]) == list(data.classes)

val_set = Subset(data, split["val_indices"])

loader = DataLoader(
    val_set,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# 5. Generate predictions from improved DINOv2
new_probs = []
new_labels = []

with torch.inference_mode():
    for images, y in tqdm(loader, desc="Improved DINOv2"):

        images = images.to(device)

        with torch.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda")
        ):
            outputs = dino_model(images)

        new_probs.append(
            torch.softmax(outputs.float(), dim=1).cpu()
        )
        new_labels.append(y)

dino_probs = torch.cat(new_probs)
check_labels = torch.cat(new_labels)

# Critical: predictions must refer to identical images
assert torch.equal(labels, check_labels)
assert conv_probs.shape == dino_probs.shape

# 6. Evaluate accuracy and cross-entropy
def metrics(probs):
    accuracy = 100 * (
        probs.argmax(1) == labels
    ).float().mean().item()

    top5 = 100 * (
        probs.topk(5, dim=1).indices == labels[:, None]
    ).any(1).float().mean().item()

    ce = F.nll_loss(
        probs.clamp_min(1e-12).log(),
        labels
    ).item()

    return accuracy, top5, ce

print("\n===== MODEL COMPARISON =====")

for name, probs in [
    ("ConvNeXt", conv_probs),
    ("Improved DINOv2", dino_probs)
]:
    acc, top5, ce = metrics(probs)
    print(f"{name}: Top-1={acc:.2f}% | Top-5={top5:.2f}% | CE={ce:.4f}")

# 7. Test ensemble weights
print("\n===== NEW ENSEMBLE =====")

results = []

for w in [0.0, 0.25, 0.50, 0.75, 1.0]:

    fused = w * conv_probs + (1 - w) * dino_probs

    acc, top5, ce = metrics(fused)
    results.append((acc, w, top5, ce))

    print(
        f"ConvNeXt {w:.0%} + DINOv2 {1-w:.0%}: "
        f"Top-1={acc:.2f}% | Top-5={top5:.2f}% | CE={ce:.4f}"
    )

best_acc, best_w, best_top5, best_ce = max(results)

print("\n===== BEST RESULT =====")
print(f"Best Top-1: {best_acc:.2f}%")
print(f"Best Top-5: {best_top5:.2f}%")
print(f"Cross-Entropy: {best_ce:.4f}")
print(f"ConvNeXt weight: {best_w:.2f}")
print(f"DINOv2 weight: {1-best_w:.2f}")

# 8. Save predictions and comparison
torch.save({
    "conv_probs": conv_probs,
    "dino_probs": dino_probs,
    "labels": labels,
    "classes": old["classes"],
    "best_weight": best_w,
    "best_accuracy": best_acc,
    "best_top5": best_top5,
    "best_cross_entropy": best_ce
}, f"{save_dir}/fusion_improved_validation.pth")

print("\nResults saved successfully!")


In [ ]:
import torch
from collections import Counter

results = torch.load(
    f"{save_dir}/fusion_improved_validation.pth",
    map_location="cpu",
    weights_only=True
)

probs = (
    0.25 * results["conv_probs"]
    + 0.75 * results["dino_probs"]
)

labels = results["labels"]
classes = results["classes"]
predictions = probs.argmax(dim=1)

# Accuracy for every food class
class_total = torch.bincount(
    labels, minlength=101
)

class_correct = torch.bincount(
    labels[predictions == labels],
    minlength=101
)

print("===== HARDEST 15 FOOD CLASSES =====")

class_accuracy = []

for i in range(101):
    if class_total[i] > 0:
        acc = 100 * (
            class_correct[i] / class_total[i]
        ).item()
        class_accuracy.append((acc, classes[i]))

for acc, name in sorted(class_accuracy)[:15]:
    print(f"{name}: {acc:.2f}%")

# Most common classification mistakes
mistakes = Counter(
    (actual, predicted)
    for actual, predicted in zip(
        labels.tolist(),
        predictions.tolist()
    )
    if actual != predicted
)

print("\n===== TOP 15 CONFUSIONS =====")

for (actual, predicted), count in mistakes.most_common(15):
    print(
        f"{classes[actual]} -> "
        f"{classes[predicted]}: {count}"
    )

## Additional model and experimental comparisons — original cell 78

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%pip install -q "huggingface_hub==1.33.0"

In [ ]:
%pip check

In [ ]:
%pip install -q jedi

In [ ]:
%pip check

In [ ]:

import torch
import timm

# 1. Model repository on Hugging Face
d3_model_id = "hf-hub:merve/food101-vit-dinov3"

# 2. Download model architecture and trained weights
d3_model = timm.create_model(
    d3_model_id,
    pretrained=True
)

# 3. Move model to GPU
d3_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
d3_model = d3_model.to(d3_device)

# 4. Switch to inference mode
d3_model.eval()

# 5. Read the required image preprocessing settings
d3_config = timm.data.resolve_model_data_config(d3_model)

# 6. Display model information
d3_total_params = sum(p.numel() for p in d3_model.parameters())

print("===== DINOv3 INFORMATION =====")
print("Model loaded successfully!")
print("Device:", d3_device)
print("Classes:", d3_model.num_classes)
print("Parameters:", f"{d3_total_params:,}")
print("Input size:", d3_config["input_size"])
print("Mean:", d3_config["mean"])
print("Std:", d3_config["std"])




In [ ]:

import torch
from transformers import (
    AutoConfig,
    AutoImageProcessor,
    AutoModelForImageClassification
)

# 1. Repository of the pretrained Food-101 model
d3_repo = "merve/food101-vit-dinov3"

# 2. Download model configuration
d3_hf_config = AutoConfig.from_pretrained(d3_repo)

# Some exported models have model_args=None
if getattr(d3_hf_config, "model_args", None) is None:
    d3_hf_config.model_args = {}

# 3. Download the image preprocessing configuration
d3_processor = AutoImageProcessor.from_pretrained(d3_repo)

# 4. Download model architecture and trained weights
d3_model = AutoModelForImageClassification.from_pretrained(
    d3_repo,
    config=d3_hf_config
)

# 5. Move model to GPU and enable evaluation mode
d3_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

d3_model = d3_model.to(d3_device)
d3_model.eval()

# 6. Inspect the loaded model
d3_params = sum(
    p.numel() for p in d3_model.parameters()
)

print("\n===== DINOv3 INFORMATION =====")
print("Model loaded successfully!")
print("Device:", d3_device)
print("Architecture:", d3_model.config.architecture)
print("Classes:", d3_model.config.num_labels)
print("Parameters:", f"{d3_params:,}")
print("Processor:", type(d3_processor).__name__)


In [ ]:

import io
import torch

from PIL import Image
from google.colab import files
from IPython.display import display

# 1. Upload one food image from your computer
uploaded = files.upload()

image_name = next(iter(uploaded))

# 2. Open the image and convert it to RGB
food_image = Image.open(
    io.BytesIO(uploaded[image_name])
).convert("RGB")

print("Uploaded image:", image_name)
display(food_image)

# 3. Prepare the image for DINOv3
d3_inputs = d3_processor(
    images=food_image,
    return_tensors="pt"
)

# 4. Move the input tensor to the same GPU
d3_inputs = {
    key: value.to(d3_device)
    for key, value in d3_inputs.items()
}

print("Input tensor shape:",
      d3_inputs["pixel_values"].shape)

# 5. Run the model without updating weights
d3_model.eval()

with torch.inference_mode():
    d3_outputs = d3_model(**d3_inputs)
    d3_logits = d3_outputs.logits

# 6. Convert raw scores into probabilities
d3_probs = torch.softmax(d3_logits, dim=1)

# 7. Find the five highest predictions
top_probs, top_indices = torch.topk(
    d3_probs[0],
    k=5
)

print("\n===== DINOv3 TOP-5 PREDICTIONS =====")

for rank, (prob, idx) in enumerate(
    zip(top_probs, top_indices),
    start=1
):
    class_id = int(idx.item())

    food_name = d3_model.config.id2label.get(
        class_id,
        f"Class {class_id}"
    )

    print(
        f"{rank}. {food_name}: "
        f"{prob.item() * 100:.2f}%"
    )


In [ ]:

import os
import random
import torch

from collections import defaultdict
from torchvision.datasets import Food101
from google.colab import drive

# 1. Connect Google Drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

d3_save_dir = "/content/drive/MyDrive/NutriVision_AI"
os.makedirs(d3_save_dir, exist_ok=True)

# 2. Load the official Food-101 test split
d3_dataset = Food101(
    root="/content/data",
    split="test",
    download=True
)

print("Total images:", len(d3_dataset))
print("Number of classes:", len(d3_dataset.classes))

# 3. Verify that label IDs match DINOv3
for i, name in enumerate(d3_dataset.classes):
    model_name = d3_model.config.id2label.get(
        i,
        d3_model.config.id2label.get(str(i), "")
    )

    assert str(model_name).lower().replace(" ", "_") == name, (
        f"Label mismatch at {i}: {name} vs {model_name}"
    )

print("Class mappings verified!")

# 4. Group image indices by their true class
d3_by_class = defaultdict(list)

for idx, label in enumerate(d3_dataset._labels):
    d3_by_class[int(label)].append(idx)

# 5. Fixed, reproducible split
rng = random.Random(42)

d3_dev_indices = []
d3_final_indices = []

for class_id in range(101):
    indices = d3_by_class[class_id].copy()
    rng.shuffle(indices)

    # 50 images per class for development
    d3_dev_indices.extend(indices[:50])

    # Remaining 200 per class for later evaluation
    d3_final_indices.extend(indices[50:])

# 6. Save split for future experiments
torch.save(
    {
        "dev_indices": d3_dev_indices,
        "final_indices": d3_final_indices,
        "classes": d3_dataset.classes,
        "seed": 42
    },
    f"{d3_save_dir}/dinov3_eval_split.pth"
)

print("\n===== DINOv3 DATA SPLIT =====")
print("Development images:", len(d3_dev_indices))
print("Reserved images:", len(d3_final_indices))
print("Split saved successfully!")


In [ ]:

import re
import torch

# 1. Normalize class names for comparison
def normalize_food_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

# 2. Read the model's actual class names
d3_model_labels = {
    int(idx): name
    for idx, name in d3_model.config.id2label.items()
}

# 3. Build a lookup: food name -> model class ID
d3_name_to_model_id = {}

for model_id, name in d3_model_labels.items():
    key = normalize_food_name(name)

    if key in d3_name_to_model_id:
        raise ValueError(f"Duplicate model class name: {name}")

    d3_name_to_model_id[key] = model_id

# 4. Verify ALL 101 classes exist in both sources
dataset_names = {
    normalize_food_name(name)
    for name in d3_dataset.classes
}

model_names = set(d3_name_to_model_id)

missing = dataset_names - model_names
extra = model_names - dataset_names

print("Missing from model:", missing)
print("Extra in model:", extra)

assert not missing and not extra, (
    "Class names are not fully compatible!"
)

assert len(d3_dataset.classes) == len(d3_model_labels) == 101

# 5. Map every Torchvision class ID to its DINOv3 ID
d3_tv_to_model = torch.tensor([
    d3_name_to_model_id[normalize_food_name(name)]
    for name in d3_dataset.classes
], dtype=torch.long)

# 6. Verify the mapping is one-to-one
assert len(set(d3_tv_to_model.tolist())) == 101

print("\n===== CLASS MAPPING VERIFIED =====")

for dataset_id in [0, 16, 17, 50, 100]:
    model_id = d3_tv_to_model[dataset_id].item()

    print(
        f"Dataset ID {dataset_id:3d} "
        f"({d3_dataset.classes[dataset_id]})"
        f" -> Model ID {model_id:3d} "
        f"({d3_model_labels[model_id]})"
    )

# 7. Convert ground-truth labels to DINOv3 IDs
d3_true_labels_model = d3_tv_to_model[
    torch.tensor(d3_dataset._labels, dtype=torch.long)
]

print("\nTotal aligned labels:", len(d3_true_labels_model))
print("Mapping completed successfully!")


In [ ]:

import os
import random
import torch
import torch.nn.functional as F

from collections import defaultdict
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# 1. Create or restore a fixed evaluation split
d3_split_path = f"{d3_save_dir}/dinov3_eval_split.pth"

if os.path.exists(d3_split_path):
    d3_split = torch.load(
        d3_split_path,
        map_location="cpu",
        weights_only=True
    )
else:
    by_class = defaultdict(list)

    for idx, label in enumerate(d3_dataset._labels):
        by_class[int(label)].append(idx)

    rng = random.Random(42)
    dev_indices = []
    final_indices = []

    for class_id in range(101):
        indices = by_class[class_id].copy()
        rng.shuffle(indices)

        dev_indices.extend(indices[:50])
        final_indices.extend(indices[50:])

    d3_split = {
        "dev_indices": dev_indices,
        "final_indices": final_indices,
        "seed": 42
    }

    torch.save(d3_split, d3_split_path)

assert len(d3_split["dev_indices"]) == 5050
assert len(d3_split["final_indices"]) == 20200

# 2. Create the development dataset
d3_dev_dataset = Subset(
    d3_dataset,
    d3_split["dev_indices"]
)

# 3. Prepare each image and align its label
def d3_collate(batch):
    images, labels = zip(*batch)

    inputs = d3_processor(
        images=list(images),
        return_tensors="pt"
    )

    # Convert Torchvision labels to DINOv3 class IDs
    labels = d3_tv_to_model[
        torch.tensor(labels, dtype=torch.long)
    ]

    return inputs["pixel_values"], labels

# 4. Create batches
d3_dev_loader = DataLoader(
    d3_dev_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    collate_fn=d3_collate
)

# 5. Evaluate pretrained DINOv3
d3_model.eval()

all_probs = []
all_labels = []

with torch.inference_mode():
    for images, labels in tqdm(
        d3_dev_loader,
        desc="DINOv3 Baseline"
    ):
        images = images.to(d3_device)

        outputs = d3_model(pixel_values=images)
        logits = outputs.logits

        probabilities = torch.softmax(
            logits.float(), dim=1
        )

        all_probs.append(probabilities.cpu())
        all_labels.append(labels.cpu())

# 6. Calculate evaluation metrics
d3_probs = torch.cat(all_probs)
d3_labels = torch.cat(all_labels)

predictions = d3_probs.argmax(dim=1)

top1 = (
    predictions == d3_labels
).float().mean().item() * 100

top5 = (
    d3_probs.topk(5, dim=1).indices
    == d3_labels[:, None]
).any(dim=1).float().mean().item() * 100

ce_loss = F.nll_loss(
    d3_probs.clamp_min(1e-12).log(),
    d3_labels
).item()

print("\n===== DINOv3 BASELINE RESULTS =====")
print(f"Top-1 Accuracy: {top1:.2f}%")
print(f"Top-5 Accuracy: {top5:.2f}%")
print(f"Cross-Entropy: {ce_loss:.4f}")
print(f"Evaluated images: {len(d3_labels)}")

# 7. Save predictions for later error analysis
torch.save({
    "probs": d3_probs,
    "labels": d3_labels,
    "indices": d3_split["dev_indices"],
    "top1": top1,
    "top5": top5,
    "cross_entropy": ce_loss
}, f"{d3_save_dir}/dinov3_baseline_dev.pth")

print("Baseline results saved!")


In [ ]:

import torch
import matplotlib.pyplot as plt
from collections import Counter

# Load the predictions we already saved
diagnostic = torch.load(
    f"{d3_save_dir}/dinov3_baseline_dev.pth",
    map_location="cpu",
    weights_only=True
)

probs = diagnostic["probs"]
labels = diagnostic["labels"]
indices = diagnostic["indices"]

preds = probs.argmax(dim=1)
confidence = probs.max(dim=1).values

def food_name(class_id):
    return d3_model.config.id2label[int(class_id)]

correct = preds == labels

print("===== DATA CHECK =====")
print("Predictions shape:", tuple(probs.shape))
print("Number of true classes:", labels.unique().numel())
print("Number of predicted classes:", preds.unique().numel())
print("Average confidence:", f"{confidence.mean().item()*100:.2f}%")
print("Correct predictions:", int(correct.sum()))
print("Wrong predictions:", int((~correct).sum()))

# Check that image indices and labels are aligned
for pos in [0, 505, 1010, 1515, 2020, 2525]:
    dataset_index = indices[pos]
    _, tv_label = d3_dataset[dataset_index]

    assert d3_tv_to_model[tv_label].item() == labels[pos].item()

print("Image/label alignment: PASSED")

print("\n===== MOST PREDICTED CLASSES =====")
counts = Counter(preds.tolist())

for class_id, count in counts.most_common(10):
    print(f"{food_name(class_id)}: {count} images")

print("\n===== MOST COMMON WRONG PREDICTIONS =====")
pairs = Counter(
    (int(true), int(pred))
    for true, pred in zip(labels, preds)
    if true != pred
)

for (true, pred), count in pairs.most_common(10):
    print(f"{food_name(true)} -> {food_name(pred)}: {count}")

# View real images with their predictions
print("\n===== SAMPLE IMAGES =====")

sample_positions = [0, 505, 1010, 1515, 2020, 2525]

fig, axes = plt.subplots(2, 3, figsize=(13, 9))

for ax, pos in zip(axes.flat, sample_positions):
    image, _ = d3_dataset[indices[pos]]

    true_name = food_name(labels[pos])
    pred_name = food_name(preds[pos])
    conf = confidence[pos].item() * 100

    ax.imshow(image)
    ax.set_title(
        f"True: {true_name}\n"
        f"Predicted: {pred_name} ({conf:.1f}%)",
        fontsize=10
    )
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import os
import torch
from google.colab import drive

# 1. Connect Google Drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

d3_save_dir = "/content/drive/MyDrive/NutriVision_AI"

# 2. Load our saved evaluation results
path = f"{d3_save_dir}/dinov3_baseline_dev.pth"

data = torch.load(
    path,
    map_location="cpu",
    weights_only=True
)

probs = data["probs"]
labels = data["labels"].long()

# 3. Get the predicted class for each image
preds = probs.argmax(dim=1)

# 4. Identify all classes the model predicted
active_classes = torch.unique(preds)
all_classes = torch.arange(probs.shape[1])

inactive_classes = all_classes[
    ~torch.isin(all_classes, active_classes)
]

# 5. Separate images by their true class
active_mask = torch.isin(labels, active_classes)
correct = (preds == labels)

# 6. Measure how probabilities are distributed
active_probability = probs[:, active_classes].sum(dim=1)

print("===== DINOv3 FINAL DIAGNOSIS =====")

print("Total classes:", probs.shape[1])
print("Predicted classes:", len(active_classes))
print("Never predicted classes:", len(inactive_classes))

print("\n===== ACCURACY =====")

print(
    "Accuracy on active classes:",
    f"{correct[active_mask].float().mean().item()*100:.2f}%"
)

print(
    "Accuracy on other classes:",
    f"{correct[~active_mask].float().mean().item()*100:.2f}%"
)

print("\n===== PROBABILITY DISTRIBUTION =====")

print(
    "Average probability assigned to active classes:",
    f"{active_probability.mean().item()*100:.2f}%"
)

print(
    "Average probability assigned to other classes:",
    f"{(1-active_probability).mean().item()*100:.2f}%"
)

print("\n===== PREDICTED CLASS DETAILS =====")

for class_id in active_classes.tolist():

    predicted_count = (preds == class_id).sum().item()
    true_count = (labels == class_id).sum().item()

    correct_count = (
        (preds == class_id) &
        (labels == class_id)
    ).sum().item()

    print(
        f"Class {class_id:3d} | "
        f"True: {true_count:2d} | "
        f"Predicted: {predicted_count:4d} | "
        f"Correct: {correct_count:2d}"
    )


In [ ]:

import torch
import timm

# Official general pretrained DINOv3
d3_general_name = "vit_base_patch16_dinov3.lvd1689m"

# Load the general pretrained weights
# Replace the original head with a new 101-class head
d3_general = timm.create_model(
    d3_general_name,
    pretrained=True,
    num_classes=101
)

# We will configure training after checking the model
d3_general = d3_general.cpu()

total_params = sum(
    p.numel() for p in d3_general.parameters()
)

print("===== GENERAL DINOv3 =====")
print("Model:", d3_general_name)
print("Parameters:", f"{total_params:,}")
print("Output classes:", d3_general.num_classes)
print("Feature dimension:", d3_general.num_features)
print("Classifier:", d3_general.get_classifier())

print("\nGeneral DINOv3 loaded successfully!")


In [ ]:
import os
import torch
import timm

from torchvision.datasets import Food101
from torch.utils.data import DataLoader, Subset

# 1. Get DINOv3's required preprocessing
d3_config = timm.data.resolve_model_data_config(
    d3_general
)

d3_train_transform = timm.data.create_transform(
    **d3_config,
    is_training=True
)

d3_val_transform = timm.data.create_transform(
    **d3_config,
    is_training=False
)

# 2. Load Food-101 training images
d3_food_train = Food101(
    root="/content/data",
    split="train",
    transform=d3_train_transform,
    download=False
)

d3_food_val = Food101(
    root="/content/data",
    split="train",
    transform=d3_val_transform,
    download=False
)

# 3. Restore our original train/validation split
d3_split = torch.load(
    "/content/drive/MyDrive/NutriVision_AI/data_split.pth",
    map_location="cpu",
    weights_only=True
)

print("Split keys:", list(d3_split.keys()))

d3_train_indices = d3_split["train_indices"]
d3_val_indices = d3_split["val_indices"]

# 4. Check that the split is correct
assert len(d3_train_indices) == 68175
assert len(d3_val_indices) == 7575

assert set(d3_train_indices).isdisjoint(
    set(d3_val_indices)
)

assert d3_food_train.classes == d3_food_val.classes
assert len(d3_food_train.classes) == 101

# 5. Create the datasets
d3_train_dataset = Subset(
    d3_food_train,
    d3_train_indices
)

d3_val_dataset = Subset(
    d3_food_val,
    d3_val_indices
)

# 6. Create DataLoaders
d3_train_loader = DataLoader(
    d3_train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

d3_val_loader = DataLoader(
    d3_val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# 7. Check one batch before training
images, labels = next(iter(d3_train_loader))

print("\n===== DINOv3 DATA READY =====")
print("Training images:", len(d3_train_dataset))
print("Validation images:", len(d3_val_dataset))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Input size:", d3_config["input_size"])
print("Number of classes:", len(d3_food_train.classes))

In [ ]:
import os

paths = [
    "/content/data/food-101",
    "/content/data/food-101/images",
    "/content/data/food-101/meta/train.json",
    "/content/data/food-101/meta/test.json"
]

for path in paths:
    print(os.path.exists(path), path)

In [ ]:

from torchvision.datasets import Food101

# Training images
d3_food_train = Food101(
    root="/content/data",
    split="train",
    transform=d3_train_transform,
    download=True
)

# Validation images with evaluation preprocessing
d3_food_val = Food101(
    root="/content/data",
    split="train",
    transform=d3_val_transform,
    download=False
)

print("Training dataset:", len(d3_food_train))
print("Validation source:", len(d3_food_val))
print("Number of classes:", len(d3_food_train.classes))

In [ ]:

import os
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# ======================================
# 1. Configure device and model
# ======================================

d3_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

d3_general = d3_general.to(d3_device)

# Freeze all pretrained parameters
for param in d3_general.parameters():
    param.requires_grad = False

# Unfreeze only the new classification head
d3_head = d3_general.get_classifier()

for param in d3_head.parameters():
    param.requires_grad = True

trainable = sum(
    p.numel()
    for p in d3_general.parameters()
    if p.requires_grad
)

print("Trainable parameters:", f"{trainable:,}")

# ======================================
# 2. Loss function and optimizer
# ======================================

train_criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

val_criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    d3_head.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

use_bf16 = (
    d3_device.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

# ======================================
# 3. Train the classification head
# ======================================

# Keep the frozen backbone in evaluation mode
d3_general.eval()
d3_head.train()

train_correct = 0
train_total = 0
train_loss_sum = 0.0

progress = tqdm(
    d3_train_loader,
    desc="DINOv3 Head Warm-up"
)

for images, labels in progress:

    images = images.to(
        d3_device,
        non_blocking=True
    )

    labels = labels.to(
        d3_device,
        non_blocking=True
    )

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type=d3_device.type,
        dtype=torch.bfloat16,
        enabled=use_bf16
    ):
        outputs = d3_general(images)
        loss = train_criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    batch_size = labels.size(0)

    train_loss_sum += loss.item() * batch_size
    train_correct += (
        outputs.argmax(dim=1) == labels
    ).sum().item()
    train_total += batch_size

    progress.set_postfix(
        loss=f"{loss.item():.3f}",
        acc=f"{100*train_correct/train_total:.2f}%"
    )

train_acc = 100 * train_correct / train_total
train_loss = train_loss_sum / train_total

# ======================================
# 4. Validation
# ======================================

d3_general.eval()

val_correct = 0
val_top5_correct = 0
val_total = 0
val_loss_sum = 0.0

with torch.inference_mode():

    for images, labels in tqdm(
        d3_val_loader,
        desc="DINOv3 Validation"
    ):

        images = images.to(
            d3_device,
            non_blocking=True
        )

        labels = labels.to(
            d3_device,
            non_blocking=True
        )

        with torch.autocast(
            device_type=d3_device.type,
            dtype=torch.bfloat16,
            enabled=use_bf16
        ):
            outputs = d3_general(images)

        loss = val_criterion(
            outputs.float(), labels
        )

        val_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()

        val_top5_correct += (
            outputs.topk(5, dim=1).indices
            == labels[:, None]
        ).any(dim=1).sum().item()

        val_loss_sum += loss.item() * labels.size(0)
        val_total += labels.size(0)

val_acc = 100 * val_correct / val_total
val_top5 = 100 * val_top5_correct / val_total
val_loss = val_loss_sum / val_total

# ======================================
# 5. Save model to Google Drive
# ======================================

d3_head_path = (
    "/content/drive/MyDrive/NutriVision_AI/"
    "dinov3_head_warmup.pth"
)

torch.save(
    {
        "model_state_dict": d3_general.state_dict(),
        "architecture": "vit_base_patch16_dinov3.lvd1689m",
        "num_classes": 101,
        "class_names": d3_food_train.classes,
        "train_accuracy": train_acc,
        "val_accuracy": val_acc,
        "val_top5": val_top5,
        "val_loss": val_loss,
        "stage": "head_warmup",
        "epoch": 1
    },
    d3_head_path
)

print("\n===== DINOv3 HEAD WARM-UP =====")
print(f"Train Accuracy: {train_acc:.2f}%")
print(f"Train Loss: {train_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.2f}%")
print(f"Validation Top-5: {val_top5:.2f}%")
print(f"Validation Loss: {val_loss:.4f}")
print("Checkpoint saved:", d3_head_path)


In [ ]:

from pathlib import Path
import subprocess

repo_path = Path("/content/NoisyViT_Food")

# Download the research source code only
if not (repo_path / "ViT.py").exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/Tonmoy-Ghosh/NoisyViT_Food.git",
            str(repo_path)
        ],
        check=True
    )

print("===== NOISYVIT REPOSITORY =====")

for filename in [
    "ViT.py",
    "Inference.py",
    "parameters.py",
    "runScript.txt",
    "requirements.txt"
]:
    path = repo_path / filename
    print(filename, "FOUND" if path.exists() else "MISSING")

print("\n===== FOOD-101 CONFIGURATION =====")

script = (repo_path / "runScript.txt").read_text()

for line in script.splitlines():
    if "--datasets food101" in line:
        print(line)

print("\nSource code ready!")


In [ ]:

import os
import torch
from pathlib import Path
from google.colab import drive

# Connect to Google Drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

save_dir = Path(
    "/content/drive/MyDrive/NutriVision_AI"
)

# Check our original model files
files_to_check = [
    "dinov2_last2_best.pth",
    "convnext_stage3_best.pth",
    "fusion_improved_validation.pth",
    "data_split.pth"
]

print("===== NUTRIVISION CHECKPOINTS =====")

for filename in files_to_check:
    path = save_dir / filename

    if path.exists():
        size = path.stat().st_size / (1024 ** 2)
        print(f"FOUND: {filename} ({size:.1f} MB)")
    else:
        print(f"MISSING: {filename}")

print("\n===== GPU =====")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU unavailable")

# Check our saved ensemble result
fusion_path = save_dir / "fusion_improved_validation.pth"

if fusion_path.exists():
    fusion = torch.load(
        fusion_path,
        map_location="cpu",
        weights_only=True
    )

    print("\n===== PREVIOUS BEST ENSEMBLE =====")
    print("Accuracy:", fusion["best_accuracy"])
    print("ConvNeXt weight:", fusion["best_weight"])
    print("DINOv2 weight:", 1 - fusion["best_weight"])


## DINOv2 final last-four-block fine-tuning — original cell 98

In [ ]:

import gc
import torch
import timm

from pathlib import Path
from torchvision import datasets, transforms as T
from torch.utils.data import DataLoader, Subset

# 1. Free memory occupied by experiments we cancelled
for name in ("d3_model", "d3_general", "dino_model", "fusion_cnn"):
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda")
save_dir = Path("/content/drive/MyDrive/NutriVision_AI")

# 2. Restore our best DINOv2 checkpoint
checkpoint = torch.load(
    save_dir / "dinov2_last2_best.pth",
    map_location="cpu",
    weights_only=True
)

d2_model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    num_classes=101,
    img_size=280
)

d2_model.load_state_dict(checkpoint["model_state_dict"])
d2_model = d2_model.to(device)

# 3. Freeze all parameters first
for p in d2_model.parameters():
    p.requires_grad = False

# 4. Unfreeze the last FOUR Transformer blocks
for block in d2_model.blocks[-4:]:
    for p in block.parameters():
        p.requires_grad = True

# Also train the normalization and classification layers
for module in (d2_model.norm, d2_model.head):
    for p in module.parameters():
        p.requires_grad = True

# 5. Recreate our DINOv2 image preprocessing
config = timm.data.resolve_model_data_config(d2_model)

mean = config["mean"]
std = config["std"]

train_transform = T.Compose([
    T.Resize(320, interpolation=T.InterpolationMode.BICUBIC),
    T.RandomCrop(280),
    T.RandomHorizontalFlip(),
    T.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1
    ),
    T.ToTensor(),
    T.Normalize(mean, std)
])

val_transform = T.Compose([
    T.Resize(320, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(280),
    T.ToTensor(),
    T.Normalize(mean, std)
])

# 6. Load the SAME Food-101 split as our old experiments
train_source = datasets.Food101(
    root="/content/data",
    split="train",
    transform=train_transform,
    download=True
)

val_source = datasets.Food101(
    root="/content/data",
    split="train",
    transform=val_transform,
    download=False
)

split = torch.load(
    save_dir / "data_split.pth",
    map_location="cpu",
    weights_only=True
)

assert len(split["train_indices"]) == 68175
assert len(split["val_indices"]) == 7575

assert set(split["train_indices"]).isdisjoint(
    set(split["val_indices"])
)

if "classes" in checkpoint:
    assert list(checkpoint["classes"]) == list(train_source.classes)

train_dataset = Subset(
    train_source,
    split["train_indices"]
)

val_dataset = Subset(
    val_source,
    split["val_indices"]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

trainable = sum(
    p.numel() for p in d2_model.parameters()
    if p.requires_grad
)

print("===== DINOv2 READY FOR IMPROVEMENT =====")
print("GPU:", torch.cuda.get_device_name(0))
print("Trainable parameters:", f"{trainable:,}")
print("Transformer blocks:", len(d2_model.blocks))
print("Unfrozen blocks:", 4)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Image resolution:", 280)


In [ ]:

import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# 1. Different learning rates for different parts
optimizer = torch.optim.AdamW([
    {
        "params": list(d2_model.blocks[-4].parameters())
                  + list(d2_model.blocks[-3].parameters()),
        "lr": 3e-6,
        "weight_decay": 0.02
    },
    {
        "params": list(d2_model.blocks[-2].parameters())
                  + list(d2_model.blocks[-1].parameters()),
        "lr": 6e-6,
        "weight_decay": 0.02
    },
    {
        "params": d2_model.norm.parameters(),
        "lr": 1e-5,
        "weight_decay": 0.0
    },
    {
        "params": d2_model.head.parameters(),
        "lr": 3e-5,
        "weight_decay": 0.01
    }
])

train_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
val_criterion = nn.CrossEntropyLoss()

epochs = 3
accumulation_steps = 2

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-7
)

assert torch.cuda.is_bf16_supported()

# Compare against our actual old ensemble
old_fusion = torch.load(
    save_dir / "fusion_improved_validation.pth",
    map_location="cpu",
    weights_only=True
)

best_ensemble = float(old_fusion["best_accuracy"])
best_single = 91.64
bad_epochs = 0

# 2. Training
for epoch in range(epochs):
    d2_model.train()
    optimizer.zero_grad(set_to_none=True)

    train_correct = 0
    train_total = 0

    for step, (images, labels) in enumerate(
        tqdm(train_loader, desc=f"Fine-tune {epoch+1}")
    ):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = d2_model(images)
            loss = train_criterion(logits.float(), labels)

        remaining = len(train_loader) - (
            step // accumulation_steps
        ) * accumulation_steps

        group_size = min(accumulation_steps, remaining)

        (loss / group_size).backward()

        if (
            (step + 1) % accumulation_steps == 0
            or step + 1 == len(train_loader)
        ):
            torch.nn.utils.clip_grad_norm_(
                d2_model.parameters(), 1.0
            )

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        train_correct += (
            logits.argmax(1) == labels
        ).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    # 3. Validation
    d2_model.eval()

    val_probs = []
    val_labels = []
    val_loss_sum = 0.0

    with torch.inference_mode():
        for images, labels in tqdm(
            val_loader,
            desc=f"Validate {epoch+1}"
        ):
            images = images.to(device)

            with torch.autocast("cuda", dtype=torch.bfloat16):
                logits = d2_model(images)

            loss = val_criterion(
                logits.float(),
                labels.to(device)
            )

            val_loss_sum += loss.item() * labels.size(0)

            val_probs.append(
                torch.softmax(logits.float(), dim=1).cpu()
            )

            val_labels.append(labels.cpu())

    probs = torch.cat(val_probs)
    labels = torch.cat(val_labels)

    # Check that old/new predictions are aligned
    assert torch.equal(labels, old_fusion["labels"])

    single_acc = (
        probs.argmax(1) == labels
    ).float().mean().item() * 100

    val_loss = val_loss_sum / len(labels)

    # 4. Compare using our ORIGINAL ensemble weights
    fused = (
        0.25 * old_fusion["conv_probs"]
        + 0.75 * probs
    )

    ensemble_acc = (
        fused.argmax(1) == labels
    ).float().mean().item() * 100

    ensemble_top5 = (
        fused.topk(5, dim=1).indices
        == labels[:, None]
    ).any(1).float().mean().item() * 100

    print(f"\n===== EPOCH {epoch+1}/{epochs} =====")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"DINOv2 Validation: {single_acc:.2f}%")
    print(f"Ensemble Validation: {ensemble_acc:.2f}%")
    print(f"Ensemble Top-5: {ensemble_top5:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")

    # 5. Save if the overall system improves
    if ensemble_acc > best_ensemble + 0.001:
        best_ensemble = ensemble_acc
        bad_epochs = 0

        torch.save({
            "model_state_dict": d2_model.state_dict(),
            "epoch": epoch + 1,
            "single_accuracy": single_acc,
            "ensemble_accuracy": ensemble_acc,
            "class_names": train_source.classes
        }, save_dir / "dinov2_last4_best.pth")

        torch.save({
            "dino_probs": probs,
            "conv_probs": old_fusion["conv_probs"],
            "labels": labels,
            "ensemble_accuracy": ensemble_acc
        }, save_dir / "dinov2_last4_fusion_best.pth")

        print("NEW BEST ENSEMBLE SAVED!")

    else:
        bad_epochs += 1
        print("No ensemble improvement.")

    scheduler.step()

    print(f"Best Ensemble: {best_ensemble:.2f}%")

    if bad_epochs >= 2:
        print("Early stopping.")
        break

print("\nExperiment completed.")
print(f"Best Ensemble Accuracy: {best_ensemble:.2f}%")


In [ ]:

import gc
import torch
import timm
import torch.nn.functional as F

from pathlib import Path
from torchvision.datasets import Food101
from torchvision import transforms as T
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

root = Path("/content/drive/MyDrive/NutriVision_AI")
device = torch.device("cuda")

# 1. Restore the clean development image indices
split_file = root / "dinov3_eval_split.pth"

split = torch.load(
    split_file,
    map_location="cpu",
    weights_only=True
)

indices = split["dev_indices"]
assert len(indices) == 5050

# 2. Use the original DINOv2 preprocessing
dino_transform = T.Compose([
    T.Resize(320, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(280),
    T.ToTensor(),
    T.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# 3. Reusable evaluation function
def evaluate_model(model, transform, name):

    dataset = Food101(
        root="/content/data",
        split="test",
        transform=transform,
        download=False
    )

    loader = DataLoader(
        Subset(dataset, indices),
        batch_size=16,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    all_probs = []
    all_labels = []

    model = model.to(device).eval()

    with torch.inference_mode():
        for images, labels in tqdm(loader, desc=name):

            images = images.to(device)

            with torch.autocast("cuda", dtype=torch.bfloat16):
                logits = model(images)

            all_probs.append(
                torch.softmax(logits.float(), dim=1).cpu()
            )

            all_labels.append(labels)

    probs = torch.cat(all_probs)
    labels = torch.cat(all_labels)

    return probs, labels, dataset.classes

# 4. Evaluate the best improved DINOv2
print("Loading DINOv2...")

dino = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    num_classes=101,
    img_size=280
)

dino_path = root / "dinov2_last4_best.pth"

dino_ckpt = torch.load(
    dino_path,
    map_location="cpu",
    weights_only=True
)

dino.load_state_dict(dino_ckpt["model_state_dict"])

dino_probs, labels, classes = evaluate_model(
    dino,
    dino_transform,
    "DINOv2 Clean Development"
)

assert list(dino_ckpt["class_names"]) == list(classes)

del dino, dino_ckpt
gc.collect()
torch.cuda.empty_cache()

# 5. Evaluate ConvNeXt using its model preprocessing
print("Loading ConvNeXt...")

conv = timm.create_model(
    "convnext_base.fb_in22k_ft_in1k_384",
    pretrained=False,
    num_classes=101
)

conv_ckpt = torch.load(
    root / "convnext_stage3_best.pth",
    map_location="cpu",
    weights_only=True
)

conv_state = conv_ckpt.get(
    "model_state_dict",
    conv_ckpt.get("state_dict")
)

if conv_state is None:
    raise KeyError("ConvNeXt checkpoint state key not found")

conv.load_state_dict(conv_state)

conv_cfg = timm.data.resolve_model_data_config(conv)

conv_transform = timm.data.create_transform(
    **conv_cfg,
    is_training=False
)

conv_probs, conv_labels, conv_classes = evaluate_model(
    conv,
    conv_transform,
    "ConvNeXt Clean Development"
)

assert torch.equal(labels, conv_labels)
assert classes == conv_classes

del conv, conv_ckpt
gc.collect()
torch.cuda.empty_cache()

# 6. Combine using our previously selected weights
fusion_probs = 0.75 * dino_probs + 0.25 * conv_probs

def report(name, probs):
    correct = (probs.argmax(1) == labels)
    top1 = correct.float().mean().item() * 100

    top5 = (
        probs.topk(5, dim=1).indices == labels[:, None]
    ).any(dim=1).float().mean().item() * 100

    loss = F.nll_loss(
        probs.clamp_min(1e-12).log(),
        labels
    ).item()

    print(
        f"{name}: Top-1={top1:.2f}% | "
        f"Top-5={top5:.2f}% | CE={loss:.4f}"
    )

    return top1

print("\n===== CLEAN DEVELOPMENT RESULTS =====")

report("DINOv2", dino_probs)
report("ConvNeXt", conv_probs)
fusion_acc = report("Ensemble", fusion_probs)

# 7. Save predictions; no need to repeat inference
torch.save({
    "dino_probs": dino_probs,
    "conv_probs": conv_probs,
    "labels": labels,
    "indices": indices,
    "classes": classes,
    "ensemble_accuracy": fusion_acc
}, root / "ensemble_clean_development.pth")

print("\nPredictions saved successfully!")


## Additional reranking and model explorations — original cell 101

In [ ]:

import torch
from collections import Counter

path = (
    "/content/drive/MyDrive/NutriVision_AI/"
    "ensemble_clean_development.pth"
)

data = torch.load(
    path,
    map_location="cpu",
    weights_only=True
)

dino_probs = data["dino_probs"]
conv_probs = data["conv_probs"]
labels = data["labels"]
classes = data["classes"]

# Predictions of each model
dino_pred = dino_probs.argmax(dim=1)
conv_pred = conv_probs.argmax(dim=1)

# Our current ensemble
fusion_probs = 0.75 * dino_probs + 0.25 * conv_probs
fusion_pred = fusion_probs.argmax(dim=1)

dino_correct = dino_pred == labels
conv_correct = conv_pred == labels
fusion_correct = fusion_pred == labels

# How the models complement one another
both_correct = (dino_correct & conv_correct).sum().item()
only_dino = (dino_correct & ~conv_correct).sum().item()
only_conv = (~dino_correct & conv_correct).sum().item()
both_wrong = (~dino_correct & ~conv_correct).sum().item()

print("===== ENSEMBLE ERROR ANALYSIS =====")
print("Both correct:", both_correct)
print("Only DINOv2 correct:", only_dino)
print("Only ConvNeXt correct:", only_conv)
print("Both wrong:", both_wrong)

# Ideal selection between existing model top-1 answers
oracle_acc = 100 * (
    dino_correct | conv_correct
).float().mean().item()

print(f"\nIdeal model selector: {oracle_acc:.2f}%")
print(
    "Current ensemble:",
    f"{fusion_correct.float().mean().item()*100:.2f}%"
)

# Identify the hardest classes
class_errors = []

for class_id, name in enumerate(classes):
    mask = labels == class_id
    accuracy = fusion_correct[mask].float().mean().item() * 100
    class_errors.append((accuracy, name))

print("\n===== HARDEST 15 CLASSES =====")
for accuracy, name in sorted(class_errors)[:15]:
    print(f"{name}: {accuracy:.2f}%")

# Most frequent confusions
mistakes = Counter(
    (int(y), int(p))
    for y, p in zip(labels, fusion_pred)
    if y != p
)

print("\n===== TOP 15 CONFUSIONS =====")
for (actual, predicted), count in mistakes.most_common(15):
    print(
        f"{classes[actual]} -> "
        f"{classes[predicted]}: {count}"
    )


In [ ]:

import torch
from pathlib import Path

root = Path("/content/drive/MyDrive/NutriVision_AI")

data = torch.load(
    root / "ensemble_clean_development.pth",
    map_location="cpu",
    weights_only=True
)

dino = data["dino_probs"].float()
conv = data["conv_probs"].float()
labels = data["labels"].long()

fusion = 0.75 * dino + 0.25 * conv

# Top-1 predictions
dino_pred = dino.argmax(dim=1)
conv_pred = conv.argmax(dim=1)
fusion_pred = fusion.argmax(dim=1)

dino_correct = dino_pred == labels
conv_correct = conv_pred == labels
fusion_correct = fusion_pred == labels

# Images where both models are wrong
both_wrong = ~dino_correct & ~conv_correct

# Top-5 candidates from each model
dino_top5 = dino.topk(5, dim=1).indices
conv_top5 = conv.topk(5, dim=1).indices
fusion_top5 = fusion.topk(5, dim=1).indices

# Is the correct class in the candidate list?
dino_has_answer = (
    dino_top5 == labels[:, None]
).any(dim=1)

conv_has_answer = (
    conv_top5 == labels[:, None]
).any(dim=1)

fusion_has_answer = (
    fusion_top5 == labels[:, None]
).any(dim=1)

either_has_answer = dino_has_answer | conv_has_answer

# Current errors and target
total = len(labels)
current_correct = int(fusion_correct.sum())
current_errors = total - current_correct

target_correct = int(0.98 * total + 0.999999)
needed = target_correct - current_correct

print("===== TARGET ANALYSIS =====")
print("Total images:", total)
print("Current correct:", current_correct)
print("Current errors:", current_errors)
print("Additional correct needed for 98%:", needed)

print("\n===== ENSEMBLE WRONG PREDICTIONS =====")
wrong = ~fusion_correct

print("Wrong Top-1:", int(wrong.sum()))
print(
    "Correct label still in Fusion Top-5:",
    int((wrong & fusion_has_answer).sum())
)
print(
    "Correct label missing from Fusion Top-5:",
    int((wrong & ~fusion_has_answer).sum())
)

print("\n===== BOTH MODELS WRONG =====")
print("Both wrong:", int(both_wrong.sum()))
print(
    "Correct label in DINOv2 Top-5:",
    int((both_wrong & dino_has_answer).sum())
)
print(
    "Correct label in ConvNeXt Top-5:",
    int((both_wrong & conv_has_answer).sum())
)
print(
    "Correct label in either Top-5:",
    int((both_wrong & either_has_answer).sum())
)
print(
    "Correct label absent from both Top-5:",
    int((both_wrong & ~either_has_answer).sum())
)

print("\nAnalysis completed. No GPU training used.")


In [ ]:

import numpy as np
import torch
import joblib

from pathlib import Path
from sklearn.linear_model import LogisticRegression

root = Path("/content/drive/MyDrive/NutriVision_AI")

# 1. Old validation predictions for training the stacker
train = torch.load(
    root / "dinov2_last4_fusion_best.pth",
    map_location="cpu",
    weights_only=True
)

# 2. Clean development predictions for evaluation
dev = torch.load(
    root / "ensemble_clean_development.pth",
    map_location="cpu",
    weights_only=True
)

def make_features(data):
    dino = data["dino_probs"].float().numpy()
    conv = data["conv_probs"].float().numpy()

    # Use both models' entire probability distributions
    features = np.concatenate([
        np.log(np.clip(dino, 1e-8, 1.0)),
        np.log(np.clip(conv, 1e-8, 1.0))
    ], axis=1)

    return features

X_train = make_features(train)
y_train = train["labels"].long().numpy()

X_dev = make_features(dev)
y_dev = dev["labels"].long().numpy()

assert X_train.shape == (7575, 202)
assert X_dev.shape == (5050, 202)

# 3. Lightweight regularized meta-classifier
stacker = LogisticRegression(
    C=0.1,
    solver="lbfgs",
    max_iter=250
)

print("Training lightweight stacker...")
stacker.fit(X_train, y_train)

assert np.array_equal(
    stacker.classes_,
    np.arange(101)
)

# 4. Stacker probabilities
stack_probs = stacker.predict_proba(X_dev)

# Our original 75/25 ensemble
base_probs = (
    0.75 * dev["dino_probs"].float().numpy()
    + 0.25 * dev["conv_probs"].float().numpy()
)

def accuracy(probs):
    return 100 * np.mean(
        probs.argmax(axis=1) == y_dev
    )

base_accuracy = accuracy(base_probs)

print("\n===== STACKING COMPARISON =====")
print(f"Original Ensemble: {base_accuracy:.2f}%")
print(f"Pure Stacking: {accuracy(stack_probs):.2f}%")

# 5. Conservative blends
best_acc = base_accuracy
best_alpha = 0.0

for alpha in [0.25, 0.50, 0.75, 1.0]:
    blended = (
        (1 - alpha) * base_probs
        + alpha * stack_probs
    )

    acc = accuracy(blended)

    print(
        f"Stacking weight {alpha:.2f}: "
        f"{acc:.2f}%"
    )

    if acc > best_acc:
        best_acc = acc
        best_alpha = alpha

print("\n===== FINAL DECISION =====")
print(f"Baseline: {base_accuracy:.2f}%")
print(f"Best Accuracy: {best_acc:.2f}%")
print(f"Improvement: {best_acc-base_accuracy:+.2f} points")

# Save only if improvement is meaningful
if best_acc >= base_accuracy + 0.20:
    joblib.dump({
        "model": stacker,
        "alpha": best_alpha
    }, root / "stacker_best.joblib")

    print("Improved stacker saved!")
else:
    print("No meaningful gain. Keep original ensemble.")


In [ ]:

import gc
import torch
import torch.nn.functional as F
from pathlib import Path
from torchvision.datasets import Food101
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoProcessor
from tqdm.auto import tqdm

root = Path("/content/drive/MyDrive/NutriVision_AI")
dev = torch.load(
    root / "ensemble_clean_development.pth",
    map_location="cpu",
    weights_only=True
)

base = (
    0.75 * dev["dino_probs"].float()
    + 0.25 * dev["conv_probs"].float()
)
labels = dev["labels"].long()
classes = dev["classes"]

baseline = (base.argmax(1) == labels).float().mean().item() * 100

# Only investigate images with close competing predictions
top2 = base.topk(2, dim=1).values
margin = top2[:, 0] - top2[:, 1]
selected = torch.where(margin < 0.30)[0]

print("Baseline:", round(baseline, 2), "%")
print("Uncertain images:", len(selected), "/", len(labels))

# Load images in the exact saved prediction order
food_test = Food101(
    root="/content/data",
    split="test",
    download=False
)

def load_selected(pos):
    i = int(selected[pos])
    image, label = food_test[int(dev["indices"][i])]
    return image, label, i

class SelectedImages:
    def __len__(self):
        return len(selected)

    def __getitem__(self, pos):
        return load_selected(pos)

def collate(batch):
    return (
        [x[0] for x in batch],
        torch.tensor([x[1] for x in batch]),
        torch.tensor([x[2] for x in batch])
    )

loader = DataLoader(
    SelectedImages(),
    batch_size=16,
    shuffle=False,
    num_workers=0,
    collate_fn=collate
)

# A general pretrained visual-language model
model_id = "google/siglip2-base-patch16-256"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).to("cuda").eval()

prompts = [
    f"This is a photo of {name.replace('_', ' ')}."
    for name in classes
]

text_inputs = processor(
    text=prompts,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
).to("cuda")

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        text_features = model.get_text_features(**text_inputs)

    text_features = F.normalize(
        text_features.float(), dim=-1
    )

# Store visual similarity scores
scores = torch.full((len(labels), 101), float("nan"))

with torch.inference_mode():
    for images, batch_labels, positions in tqdm(
        loader, desc="Visual reranking"
    ):
        assert torch.equal(labels[positions], batch_labels)

        inputs = processor(
            images=images,
            return_tensors="pt"
        ).to("cuda")

        with torch.autocast("cuda", dtype=torch.bfloat16):
            image_features = model.get_image_features(**inputs)

        image_features = F.normalize(
            image_features.float(), dim=-1
        )

        scores[positions] = (
            image_features @ text_features.T
        ).cpu()

# Restrict new scores to our ensemble's Top-5
top5 = base.topk(5, dim=1).indices
selected_scores = scores[selected]

candidate_mask = torch.zeros_like(
    selected_scores, dtype=torch.bool
)
candidate_mask.scatter_(1, top5[selected], True)

rerank_probs = torch.softmax(
    selected_scores.masked_fill(
        ~candidate_mask, -1e9
    ) / 0.05,
    dim=1
)

best_acc = baseline
best_settings = None

print("\n===== VISUAL RERANKING RESULTS =====")
print(f"Original Ensemble: {baseline:.2f}%")

for threshold in [0.05, 0.10, 0.20, 0.30]:
    eligible = margin[selected] < threshold
    row_ids = selected[eligible]
    visual_probs = rerank_probs[eligible]

    for weight in [0.20, 0.40, 0.60]:
        combined = base.clone()

        combined[row_ids] = (
            (1 - weight) * base[row_ids]
            + weight * visual_probs
        )

        acc = (
            combined.argmax(1) == labels
        ).float().mean().item() * 100

        print(
            f"Margin < {threshold:.2f}, "
            f"Visual weight={weight:.2f}: {acc:.2f}%"
        )

        if acc > best_acc:
            best_acc = acc
            best_settings = (threshold, weight)

print("\n===== FINAL RESULT =====")
print(f"Baseline: {baseline:.2f}%")
print(f"Best Visual Reranking: {best_acc:.2f}%")
print(f"Improvement: {best_acc-baseline:+.2f} points")
print("Best settings:", best_settings)

# Preserve predictions without replacing our checkpoints
torch.save({
    "scores": scores,
    "selected": selected,
    "baseline": baseline,
    "best_accuracy": best_acc,
    "best_settings": best_settings,
    "model_id": model_id
}, root / "siglip2_reranking_development.pth")

del model
gc.collect()
torch.cuda.empty_cache()


In [ ]:

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# Extract the actual Tensor from model outputs
def get_embedding(output):
    if isinstance(output, torch.Tensor):
        return output

    pooled = getattr(output, "pooler_output", None)

    if pooled is None:
        raise TypeError(
            f"Cannot extract embedding from {type(output)}"
        )

    return pooled

# 1. Fix the TEXT features already computed
text_emb = F.normalize(
    get_embedding(text_features).float(),
    dim=-1
)

print("Text embeddings:", tuple(text_emb.shape))

# 2. Extract IMAGE features
scores = torch.full(
    (len(labels), len(classes)),
    float("nan")
)

model.eval()

with torch.inference_mode():
    for images, batch_labels, positions in tqdm(
        loader,
        desc="SigLIP2 Visual Reranking"
    ):
        assert torch.equal(labels[positions], batch_labels)

        image_inputs = processor(
            images=images,
            return_tensors="pt"
        ).to("cuda")

        with torch.autocast("cuda", dtype=torch.bfloat16):
            image_output = model.get_image_features(
                **image_inputs
            )

        image_emb = F.normalize(
            get_embedding(image_output).float(),
            dim=-1
        )

        assert image_emb.shape[1] == text_emb.shape[1]

        similarities = image_emb @ text_emb.T

        scores[positions] = similarities.cpu()

assert torch.isfinite(scores[selected]).all()

print("Visual similarities calculated successfully!")

# 3. Rerank only the original ensemble's Top-5
top5 = base.topk(5, dim=1).indices

candidate_mask = torch.zeros(
    (len(selected), len(classes)),
    dtype=torch.bool
)

candidate_mask.scatter_(1, top5[selected], True)

selected_scores = scores[selected]

rerank_probs = torch.softmax(
    selected_scores.masked_fill(
        ~candidate_mask, -1e9
    ) / 0.05,
    dim=1
)

# 4. Compare against our real baseline
baseline = (
    (base.argmax(1) == labels)
    .float().mean().item() * 100
)

best_acc = baseline
best_settings = None

top2 = base.topk(2, dim=1).values
margin = top2[:, 0] - top2[:, 1]

wrong_eligible = (
    (base.argmax(1) != labels)
    & (margin < 0.30)
).sum().item()

print("\n===== EXPERIMENT COVERAGE =====")
print("Selected images:", len(selected))
print("Current errors among selected:", wrong_eligible)
print(
    "Theoretical max with perfect corrections:",
    f"{baseline + 100 * wrong_eligible / len(labels):.2f}%"
)

print("\n===== VISUAL RERANKING RESULTS =====")
print(f"Original Ensemble: {baseline:.2f}%")

for threshold in [0.05, 0.10, 0.20, 0.30]:
    eligible = margin[selected] < threshold
    row_ids = selected[eligible]

    for weight in [0.20, 0.40, 0.60]:
        combined = base.clone()

        combined[row_ids] = (
            (1 - weight) * base[row_ids]
            + weight * rerank_probs[eligible]
        )

        acc = (
            (combined.argmax(1) == labels)
            .float().mean().item() * 100
        )

        print(
            f"Margin < {threshold:.2f}, "
            f"Visual weight={weight:.2f}: {acc:.2f}%"
        )

        if acc > best_acc:
            best_acc = acc
            best_settings = (threshold, weight)

print("\n===== FINAL RESULT =====")
print(f"Baseline: {baseline:.2f}%")
print(f"Best Visual Reranking: {best_acc:.2f}%")
print(f"Improvement: {best_acc-baseline:+.2f} points")
print("Best settings:", best_settings)

# 5. Save the results
torch.save({
    "scores": scores,
    "selected": selected,
    "baseline": baseline,
    "best_accuracy": best_acc,
    "best_settings": best_settings,
    "model_id": model_id
}, root / "siglip2_reranking_development.pth")

print("\nResults saved successfully!")


In [ ]:

import torch
from pathlib import Path

root = Path("/content/drive/MyDrive/NutriVision_AI")

dev = torch.load(
    root / "ensemble_clean_development.pth",
    map_location="cpu",
    weights_only=True
)

sig = torch.load(
    root / "siglip2_reranking_development.pth",
    map_location="cpu",
    weights_only=True
)

labels = dev["labels"].long()

base = (
    0.75 * dev["dino_probs"].float()
    + 0.25 * dev["conv_probs"].float()
)

scores = sig["scores"]
selected = sig["selected"].long()

# Rebuild the best SigLIP2 result
top5 = base.topk(5, dim=1).indices

mask = torch.zeros(
    (len(selected), 101),
    dtype=torch.bool
)
mask.scatter_(1, top5[selected], True)

visual_probs = torch.softmax(
    scores[selected].masked_fill(~mask, -1e9) / 0.05,
    dim=1
)

improved = base.clone()

improved[selected] = (
    0.40 * base[selected]
    + 0.60 * visual_probs
)

old_ok = base.argmax(1) == labels
new_ok = improved.argmax(1) == labels

corrected = (~old_ok & new_ok).sum().item()
harmed = (old_ok & ~new_ok).sum().item()

print("===== REAL IMPROVEMENT =====")
print("Corrected mistakes:", corrected)
print("Previously correct but now wrong:", harmed)
print("Net improvement:", corrected - harmed)

# Check which original mistakes can be fixed
wrong = ~old_ok

eligible = torch.zeros(len(labels), dtype=torch.bool)
eligible[selected] = True

has_answer = (
    top5 == labels[:, None]
).any(dim=1)

eligible_fixable = (
    wrong & eligible & has_answer
).sum().item()

eligible_missing = (
    wrong & eligible & ~has_answer
).sum().item()

outside_errors = (
    wrong & ~eligible
).sum().item()

max_correct = old_ok.sum().item() + eligible_fixable

print("\n===== ACCURACY LIMIT =====")
print("Errors outside selected images:", outside_errors)
print("Eligible errors with answer in Top-5:", eligible_fixable)
print("Eligible errors missing from Top-5:", eligible_missing)

print(
    "Ideal ceiling for current design:",
    f"{100 * max_correct / len(labels):.2f}%"
)

print(
    "Actual accuracy:",
    f"{100 * new_ok.float().mean():.2f}%"
)


In [ ]:

from huggingface_hub import list_repo_files

repo = "htetooyan/FoodClassifier"
files = list_repo_files(repo)

weights = [
    f for f in files
    if f.endswith((".pth", ".pt", ".safetensors", ".bin"))
]

print("Model:", repo)
print("Weight files:", weights)

if weights:
    print("READY: Checkpoint is available for evaluation.")
else:
    print("NOT READY: No model weights found.")


In [ ]:

import time
import statistics
import torch
import timm
import torch.nn as nn

from pathlib import Path
from huggingface_hub import hf_hub_download
from torchvision import datasets, transforms as T
from torchvision.models import efficientnet_v2_s
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

root = Path("/content/drive/MyDrive/NutriVision_AI")
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Download the existing trained weights
repo = "htetooyan/FoodClassifier"

path = hf_hub_download(
    repo_id=repo,
    filename="EfficientNetV2S_accuracy_96.pth"
)

ckpt = torch.load(
    path,
    map_location="cpu",
    weights_only=True
)

# 2. Extract the state dictionary
state = ckpt

if isinstance(ckpt, dict):
    for key in ("model_state_dict", "state_dict", "model", "net"):
        if key in ckpt and isinstance(ckpt[key], dict):
            state = ckpt[key]
            break

if not isinstance(state, dict):
    raise TypeError("Unexpected checkpoint format")

state = {
    k.removeprefix("module."): v
    for k, v in state.items()
}

# 3. Identify architecture from actual weight names
keys = list(state.keys())

if any(k.startswith("features.") for k in keys):
    model = efficientnet_v2_s(weights=None)
    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features, 101
    )
    architecture = "Torchvision EfficientNetV2-S"

elif any(k.startswith("conv_stem.") for k in keys):
    model = timm.create_model(
        "tf_efficientnetv2_s.in21k_ft_in1k",
        pretrained=False,
        num_classes=101
    )
    architecture = "timm EfficientNetV2-S"

else:
    print("Checkpoint keys:", keys[:20])
    raise RuntimeError(
        "Unknown architecture. Do not guess the model."
    )

# Require exact weight compatibility
model.load_state_dict(state, strict=True)
model = model.to(device).eval()

# 4. Restore our exact development image indices
dev = torch.load(
    root / "ensemble_clean_development.pth",
    map_location="cpu",
    weights_only=True
)

# The model card specifies 224x224;
# ImageNet normalization is an assumption to verify.
transform = T.Compose([
    T.Resize(
        256,
        interpolation=T.InterpolationMode.BICUBIC
    ),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

dataset = datasets.Food101(
    root="/content/data",
    split="test",
    transform=transform,
    download=False
)

# Verify class ordering if checkpoint includes class names
saved_classes = None

if isinstance(ckpt, dict):
    for k in ("classes", "class_names"):
        if k in ckpt:
            saved_classes = ckpt[k]
            break

if saved_classes is not None:
    assert list(saved_classes) == list(dataset.classes), (
        "Class ordering mismatch: evaluation would be invalid"
    )
else:
    print(
        "WARNING: No class mapping found in checkpoint. "
        "Assuming torchvision Food101 alphabetical order."
    )

loader = DataLoader(
    Subset(dataset, dev["indices"]),
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=(device == "cuda")
)

# 5. Evaluate all 5050 images
correct1 = 0
correct5 = 0
total = 0

with torch.inference_mode():
    for images, labels in tqdm(
        loader, desc="EfficientNetV2-S Evaluation"
    ):
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images).float()

        correct1 += (
            logits.argmax(1) == labels
        ).sum().item()

        correct5 += (
            logits.topk(5, dim=1).indices
            == labels[:, None]
        ).any(dim=1).sum().item()

        total += labels.numel()

top1 = 100 * correct1 / total
top5 = 100 * correct5 / total

print("\n===== ACCURACY RESULTS =====")
print("Architecture:", architecture)
print("Images tested:", total)
print(f"EfficientNetV2-S Top-1: {top1:.2f}%")
print(f"EfficientNetV2-S Top-5: {top5:.2f}%")
print("Our DINOv2 Top-1: 94.93%")
print("Our Ensemble Top-1: 95.29%")
print("Our Ensemble + SigLIP2: 95.80%")

# 6. Measure single-image inference time on CPU
torch.set_num_threads(4)

model = model.to("cpu").eval()
sample = next(iter(loader))[0][:1]

with torch.inference_mode():
    for _ in range(3):
        model(sample)

    times = []

    for _ in range(10):
        start = time.perf_counter()
        model(sample)
        times.append(time.perf_counter() - start)

print("\n===== CPU PERFORMANCE =====")
print(
    "Median time per image:",
    f"{statistics.median(times):.3f} seconds"
)
print("Device: CPU (4 threads)")
print("No GPU required for inference")

# 7. Preserve evaluation results
torch.save({
    "top1": top1,
    "top5": top5,
    "architecture": architecture,
    "cpu_median_seconds": statistics.median(times),
    "checkpoint_repo": repo,
    "class_mapping_verified": saved_classes is not None
}, root / "efficientnetv2_comparison.pth")

print("\nEvaluation completed!")


## Nutrition5k and regression preparation — original cell 109

In [ ]:

import torch
from pathlib import Path

root = Path("/content/drive/MyDrive/NutriVision_AI")

ckpt = torch.load(
    root / "calorie_best.pth",
    map_location="cpu",
    weights_only=True
)

print("===== CALORIE CHECKPOINT =====")

for key, value in ckpt.items():
    if key in ("model_state_dict", "state_dict"):
        print(key, ":", len(value), "weight tensors")
    elif isinstance(value, torch.Tensor):
        if value.numel() <= 10:
            print(key, ":", value.tolist())
        else:
            print(key, ": tensor", tuple(value.shape))
    elif isinstance(value, (int, float, str, bool, list, tuple, dict)):
        print(key, ":", str(value)[:300])

print("\n===== DATA FOLDERS =====")

for folder in [
    "/content/data/nutrition5k",
    "/content/nutrition5k",
    str(root / "nutrition5k")
]:
    print(folder, "FOUND" if Path(folder).exists() else "NOT FOUND")


In [ ]:

import pandas as pd
import requests

from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files

repo = "ai5labsOfficial/nutrition5k"

# Download metadata only, not the images
metadata_path = hf_hub_download(
    repo_id=repo,
    repo_type="dataset",
    filename="metadata.csv"
)

metadata = pd.read_csv(metadata_path)

# Inspect available image files
repo_files = list_repo_files(
    repo_id=repo,
    repo_type="dataset"
)

image_ids = {
    Path(f).stem
    for f in repo_files
    if f.startswith("images/") and f.endswith(".png")
}

# Read official Nutrition5k test split
url = (
    "https://raw.githubusercontent.com/"
    "google-research-datasets/Nutrition5k/main/"
    "dish_ids/splits/depth_test_ids.txt"
)

response = requests.get(url, timeout=30)
response.raise_for_status()

test_ids = set(response.text.split())

available_test = image_ids & test_ids

print("===== NUTRITION5K DATA CHECK =====")
print("Metadata rows:", len(metadata))
print("Available images:", len(image_ids))
print("Official test images:", len(test_ids))
print("Matching test images:", len(available_test))
print("Missing test images:", len(test_ids - image_ids))
print("Metadata columns:", metadata.columns.tolist())
print("Example dish:", metadata.head(1).to_dict("records"))



In [ ]:
import requests

# Correct location in the official Nutrition5k storage
url = (
    "https://storage.googleapis.com/"
    "nutrition5k_dataset/nutrition5k_dataset/"
    "dish_ids/splits/depth_test_ids.txt"
)

response = requests.get(url, timeout=30)
response.raise_for_status()

test_ids = set(response.text.split())
available_test = image_ids & test_ids

print("===== NUTRITION5K DATA CHECK =====")
print("Metadata rows:", len(metadata))
print("Available images:", len(image_ids))
print("Official depth test IDs:", len(test_ids))
print("Matching test images:", len(available_test))
print("Missing test images:", len(test_ids - image_ids))
print("Metadata columns:", metadata.columns.tolist())
print("Example image files:", [
    f for f in repo_files if f.startswith("images/")
][:3])

## DINOv2 feature extraction + Nutrition MLP training — original cell 112

In [ ]:

import requests
import numpy as np
import pandas as pd
import torch
import timm
import joblib

from pathlib import Path
from PIL import Image
from huggingface_hub import snapshot_download
from torchvision import transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from tqdm.auto import tqdm

root = Path("/content/drive/MyDrive/NutriVision_AI")
data_dir = Path("/content/data/nutrition5k")
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Download RGB images and metadata
snapshot_download(
    repo_id="ai5labsOfficial/nutrition5k",
    repo_type="dataset",
    allow_patterns=["images/*.png", "metadata.csv"],
    local_dir=str(data_dir),
    max_workers=8
)

df = pd.read_csv(data_dir / "metadata.csv")
df["dish_id"] = df["dish_id"].astype(str)

# 2. Load official train and test IDs
base_url = (
    "https://storage.googleapis.com/"
    "nutrition5k_dataset/nutrition5k_dataset/"
    "dish_ids/splits/"
)

def get_ids(filename):
    response = requests.get(base_url + filename, timeout=30)
    response.raise_for_status()
    return set(response.text.split())

official_train = get_ids("depth_train_ids.txt")
official_test = get_ids("depth_test_ids.txt")

available = {
    p.stem for p in (data_dir / "images").glob("*.png")
}

train_ids = sorted(
    set(df.dish_id) & available & official_train
)

test_ids = sorted(
    set(df.dish_id) & available & official_test
)

assert len(test_ids) == 507
assert set(train_ids).isdisjoint(test_ids)

print("Official train images:", len(train_ids))
print("Official test images:", len(test_ids))

# 3. Prepare nutrition targets
targets = [
    "total_calories",
    "total_mass_g",
    "fat_g",
    "carb_g",
    "protein_g"
]

df = df.set_index("dish_id")

# Use same visual preprocessing as our DINOv2
transform = T.Compose([
    T.Resize(320, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(280),
    T.ToTensor(),
    T.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

class NutritionImages(Dataset):
    def __init__(self, ids):
        self.ids = ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        dish_id = self.ids[idx]
        image = Image.open(
            data_dir / "images" / f"{dish_id}.png"
        ).convert("RGB")

        y = df.loc[dish_id, targets].to_numpy(
            dtype=np.float32
        )

        return transform(image), torch.tensor(y)

# 4. Load OUR trained DINOv2
ckpt = torch.load(
    root / "dinov2_last4_best.pth",
    map_location="cpu",
    weights_only=True
)

dino = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    num_classes=101,
    img_size=280
)

dino.load_state_dict(ckpt["model_state_dict"])
dino = dino.to(device).eval()

# 5. Extract features only once, without training
all_ids = train_ids + test_ids

loader = DataLoader(
    NutritionImages(all_ids),
    batch_size=16,
    shuffle=False,
    num_workers=0
)

features = []
nutrition_labels = []

with torch.inference_mode():
    for images, y in tqdm(
        loader, desc="Extracting Nutrition Features"
    ):
        images = images.to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=(device == "cuda")
        ):
            x = dino.forward_features(images)
            x = dino.forward_head(x, pre_logits=True)

        features.append(x.float().cpu())
        nutrition_labels.append(y)

X = torch.cat(features).numpy()
Y = torch.cat(nutrition_labels).numpy()

assert X.ndim == 2
assert X.shape[0] == len(all_ids)
assert np.isfinite(X).all()
assert np.isfinite(Y).all()

# Save small feature dataset permanently in Drive
torch.save({
    "features": torch.from_numpy(X),
    "targets": torch.from_numpy(Y),
    "ids": all_ids,
    "train_ids": train_ids,
    "test_ids": test_ids,
    "target_names": targets
}, root / "nutrition_dinov2_features.pth")

print("Feature shape:", X.shape)

# 6. Train/validation split, excluding official test
train_idx, val_idx = train_test_split(
    np.arange(len(train_ids)),
    test_size=0.15,
    random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X[train_idx])
X_val = scaler.transform(X[val_idx])

y_train = Y[train_idx]
y_val = Y[val_idx]

# Standardize targets using training values only
y_mean = y_train.mean(axis=0)
y_std = y_train.std(axis=0).clip(min=1e-6)

# 7. Train a lightweight five-output nutrition head
best_mae = float("inf")
best_model = None

print("\n===== NUTRITION HEAD TRAINING =====")

for alpha in [1.0, 10.0, 100.0]:
    regressor = Ridge(alpha=alpha)

    regressor.fit(
        X_train,
        (y_train - y_mean) / y_std
    )

    pred = regressor.predict(X_val) * y_std + y_mean
    pred = np.maximum(pred, 0)

    calorie_mae = mean_absolute_error(
        y_val[:, 0], pred[:, 0]
    )

    print(
        f"Ridge alpha={alpha}: "
        f"Calorie Validation MAE={calorie_mae:.2f} kcal"
    )

    if calorie_mae < best_mae:
        best_mae = calorie_mae
        best_model = regressor
        best_alpha = alpha

# 8. Save trained nutrition head
joblib.dump({
    "regressor": best_model,
    "feature_scaler": scaler,
    "target_mean": y_mean,
    "target_std": y_std,
    "target_names": targets,
    "alpha": best_alpha
}, root / "nutrition_dinov2_head.joblib")

torch.save({
    "train_ids": [train_ids[i] for i in train_idx],
    "val_ids": [train_ids[i] for i in val_idx],
    "test_ids": test_ids
}, root / "nutrition_experiment_split.pth")

print("\n===== FINAL NUTRITION RESULT =====")
print(f"Best Validation MAE: {best_mae:.2f} kcal")
print(f"Best alpha: {best_alpha}")
print("Nutrition head saved.")
print("Official test set untouched in this experiment.")


In [ ]:

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

root = Path("/content/drive/MyDrive/NutriVision_AI")

data = torch.load(
    root / "nutrition_dinov2_features.pth",
    map_location="cpu",
    weights_only=True
)

X = data["features"].float().numpy()
Y = data["targets"].float().numpy()

n_train = len(data["train_ids"])

assert X.shape == (3262, 768)
assert Y.shape == (3262, 5)
assert data["ids"][:n_train] == data["train_ids"]

# Recreate exactly the Ridge validation split
train_idx, val_idx = train_test_split(
    np.arange(n_train),
    test_size=0.15,
    random_state=42
)

# Normalize visual features using training data only
x_mean = X[train_idx].mean(axis=0)
x_std = X[train_idx].std(axis=0).clip(min=1e-6)

Xn = (X - x_mean) / x_std

# Targets: calories, mass, fat, carbs, protein
# Also learn energy density = calories / mass
density = Y[:, 0] / np.maximum(Y[:, 1], 1.0)

Z = np.column_stack([
    np.log1p(np.maximum(Y, 0)),
    np.log(np.maximum(density, 0.01))
]).astype("float32")

z_mean = Z[train_idx].mean(axis=0)
z_std = Z[train_idx].std(axis=0).clip(min=0.05)

Zn = (Z - z_mean) / z_std

train_ds = TensorDataset(
    torch.from_numpy(Xn[train_idx].astype("float32")),
    torch.from_numpy(Zn[train_idx])
)

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True
)

val_x = torch.from_numpy(
    Xn[val_idx].astype("float32")
)

val_calories = torch.from_numpy(
    Y[val_idx, 0].astype("float32")
)

# Small nonlinear nutrition network
class NutritionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(64, 6)
        )

    def forward(self, x):
        return self.net(x)

model = NutritionMLP()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=0.03
)

zm = torch.tensor(z_mean)
zs = torch.tensor(z_std)

# Convert predicted targets to real calorie values
def calorie_predictions(output):
    raw = output * zs + zm

    direct = torch.expm1(
        raw[:, 0].clamp(0, 9)
    ).clamp_min(0)

    mass = torch.expm1(
        raw[:, 1].clamp(0, 9)
    ).clamp_min(0)

    energy_density = torch.exp(
        raw[:, 5].clamp(-4, 3)
    )

    calculated = mass * energy_density

    return {
        "Direct": direct,
        "Mass x Density": calculated,
        "50/50 Blend": 0.5 * direct + 0.5 * calculated
    }

best_mae = float("inf")
best_method = None
bad_epochs = 0
patience = 12

print("===== NUTRITION MLP TRAINING =====")

for epoch in range(1, 81):
    model.train()

    for images_features, targets in train_loader:
        optimizer.zero_grad()

        outputs = model(images_features)

        loss_per_target = F.smooth_l1_loss(
            outputs,
            targets,
            reduction="none"
        )

        # Give calorie and density prediction extra weight
        weights = torch.tensor(
            [3.0, 1.0, 0.5, 0.5, 0.5, 2.0]
        )

        loss = (
            loss_per_target * weights
        ).mean()

        loss.backward()
        optimizer.step()

    model.eval()

    with torch.inference_mode():
        output = model(val_x)
        predictions = calorie_predictions(output)

        scores = {
            name: torch.abs(
                pred - val_calories
            ).mean().item()
            for name, pred in predictions.items()
        }

    method = min(scores, key=scores.get)
    val_mae = scores[method]

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d}: "
            f"Validation MAE={val_mae:.2f} kcal "
            f"({method})"
        )

    if val_mae < best_mae - 0.05:
        best_mae = val_mae
        best_method = method
        best_epoch = epoch
        bad_epochs = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "feature_mean": torch.from_numpy(x_mean),
            "feature_std": torch.from_numpy(x_std),
            "target_mean": torch.from_numpy(z_mean),
            "target_std": torch.from_numpy(z_std),
            "method": method,
            "epoch": epoch,
            "val_mae": val_mae
        }, root / "nutrition_mlp_best.pth")
    else:
        bad_epochs += 1

    if bad_epochs >= patience:
        print("Early stopping.")
        break

print("\n===== FINAL NUTRITION COMPARISON =====")
print("Ridge Validation MAE: 79.69 kcal")
print(f"MLP Validation MAE: {best_mae:.2f} kcal")
print(f"Best prediction method: {best_method}")
print(f"Best epoch: {best_epoch}")
print(f"Improvement: {79.69-best_mae:+.2f} kcal")

if best_mae < 79.69:
    print("MLP improved on Ridge!")
else:
    print("MLP did not improve. Keep Ridge baseline.")



In [ ]:

import torch
import torch.nn as nn
from pathlib import Path

root = Path("/content/drive/MyDrive/NutriVision_AI")

# Restore saved visual features and best nutrition head
data = torch.load(
    root / "nutrition_dinov2_features.pth",
    map_location="cpu",
    weights_only=True
)

ckpt = torch.load(
    root / "nutrition_mlp_best.pth",
    map_location="cpu",
    weights_only=True
)

class NutritionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(64, 6)
        )

    def forward(self, x):
        return self.net(x)

model = NutritionMLP()
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# Official test data is at the end of our saved features
n_train = len(data["train_ids"])

assert data["ids"][n_train:] == data["test_ids"]
assert len(data["test_ids"]) == 507
assert ckpt["method"] == "Direct"

X_test = data["features"][n_train:].float()
Y_test = data["targets"][n_train:].float()

# Apply the same training normalization
x_mean = ckpt["feature_mean"].float()
x_std = ckpt["feature_std"].float()

X_test = (X_test - x_mean) / x_std

with torch.inference_mode():
    output = model(X_test)

    # Reverse the target normalization
    raw = (
        output * ckpt["target_std"].float()
        + ckpt["target_mean"].float()
    )

    # Best method: Direct Calories
    predicted = torch.expm1(
        raw[:, 0].clamp(0, 9)
    ).clamp_min(0)

actual = Y_test[:, 0]

errors = predicted - actual
absolute_errors = errors.abs()

mae = absolute_errors.mean().item()
rmse = errors.square().mean().sqrt().item()
median_ae = absolute_errors.median().item()
within_50 = (absolute_errors <= 50).float().mean().item() * 100
within_100 = (absolute_errors <= 100).float().mean().item() * 100

print("===== OFFICIAL NUTRITION5K TEST =====")
print("Test images:", len(actual))
print(f"Calorie MAE: {mae:.2f} kcal")
print(f"Calorie RMSE: {rmse:.2f} kcal")
print(f"Median Absolute Error: {median_ae:.2f} kcal")
print(f"Within ±50 kcal: {within_50:.2f}%")
print(f"Within ±100 kcal: {within_100:.2f}%")

print("\n===== PREVIOUS RESNET18 TEST =====")
print("Previous MAE: 82.60 kcal")
print(f"New model MAE: {mae:.2f} kcal")
print(f"Difference: {82.60 - mae:+.2f} kcal")

print("\n===== SELECTED CHECKPOINT =====")
print("Best validation epoch:", ckpt["epoch"])
print("Validation MAE:", round(float(ckpt["val_mae"]), 2))
print("Prediction method:", ckpt["method"])

torch.save({
    "mae": mae,
    "rmse": rmse,
    "median_ae": median_ae,
    "within_50": within_50,
    "within_100": within_100,
    "predictions": predicted,
    "targets": actual,
    "test_ids": data["test_ids"]
}, root / "nutrition_mlp_test_results.pth")

print("\nTest results saved!")


## Held-out Food-101 evaluation and API prototype — original cell 115

In [ ]:

import gc
import torch
import timm
from pathlib import Path
from torchvision.datasets import Food101
from torchvision import transforms as T
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

root = Path("/content/drive/MyDrive/NutriVision_AI")
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 1. Load our existing development split
dev = torch.load(
    root / "ensemble_clean_development.pth",
    map_location="cpu",
    weights_only=True
)

dev_indices = set(int(i) for i in dev["indices"])

# 2. Reserve only the previously unused test images
test_indices = [
    i for i in range(25250)
    if i not in dev_indices
]

assert len(dev_indices) == 5050
assert len(test_indices) == 20200
assert not dev_indices.intersection(test_indices)

print("===== FINAL TEST SPLIT =====")
print("Development:", len(dev_indices))
print("Held-out Test:", len(test_indices))
print("Overlap: 0")

# 3. Shared evaluation function
def evaluate(model, transform, name):
    dataset = Food101(
        root="/content/data",
        split="test",
        transform=transform,
        download=False
    )

    assert len(dataset) == 25250

    loader = DataLoader(
        Subset(dataset, test_indices),
        batch_size=16 if device.type == "cuda" else 4,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda")
    )

    model.to(device).eval()

    all_probs = []
    all_labels = []

    with torch.inference_mode():
        for images, labels in tqdm(loader, desc=name):
            images = images.to(device)

            if device.type == "cuda":
                with torch.autocast(
                    "cuda", dtype=torch.bfloat16
                ):
                    logits = model(images)
            else:
                logits = model(images)

            all_probs.append(
                torch.softmax(
                    logits.float(), dim=1
                ).cpu()
            )
            all_labels.append(labels.cpu())

    return (
        torch.cat(all_probs),
        torch.cat(all_labels),
        dataset.classes
    )

# 4. Load our best DINOv2
dino_ckpt = torch.load(
    root / "dinov2_last4_best.pth",
    map_location="cpu",
    weights_only=True
)

dino = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    num_classes=101,
    img_size=280
)

dino.load_state_dict(
    dino_ckpt["model_state_dict"],
    strict=True
)

dino_transform = T.Compose([
    T.Resize(
        320,
        interpolation=T.InterpolationMode.BICUBIC
    ),
    T.CenterCrop(280),
    T.ToTensor(),
    T.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    )
])

dino_probs, labels, classes = evaluate(
    dino, dino_transform, "DINOv2 Final Test"
)

assert list(dino_ckpt["class_names"]) == list(classes)

del dino, dino_ckpt
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

# 5. Load our ConvNeXt
conv_ckpt = torch.load(
    root / "convnext_stage3_best.pth",
    map_location="cpu",
    weights_only=True
)

conv = timm.create_model(
    "convnext_base.fb_in22k_ft_in1k_384",
    pretrained=False,
    num_classes=101
)

conv_state = conv_ckpt.get(
    "model_state_dict",
    conv_ckpt.get("state_dict")
)

assert conv_state is not None, "ConvNeXt weights missing"

conv.load_state_dict(conv_state, strict=True)

conv_config = timm.data.resolve_model_data_config(conv)

conv_transform = timm.data.create_transform(
    **conv_config,
    is_training=False
)

conv_probs, conv_labels, conv_classes = evaluate(
    conv, conv_transform, "ConvNeXt Final Test"
)

assert torch.equal(labels, conv_labels)
assert classes == conv_classes

del conv, conv_ckpt
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

# 6. Apply the FROZEN ensemble weights
fusion = 0.75 * dino_probs + 0.25 * conv_probs

def show_results(name, probs):
    top1 = (
        probs.argmax(dim=1) == labels
    ).float().mean().item() * 100

    top5 = (
        probs.topk(5, dim=1).indices
        == labels[:, None]
    ).any(dim=1).float().mean().item() * 100

    print(f"{name}: Top-1={top1:.2f}% | Top-5={top5:.2f}%")
    return top1, top5

print("\n===== FINAL HELD-OUT RESULTS =====")

dino_result = show_results("DINOv2", dino_probs)
conv_result = show_results("ConvNeXt", conv_probs)
fusion_result = show_results("FINAL ENSEMBLE", fusion)

# 7. Save the final evaluation evidence
torch.save({
    "test_indices": test_indices,
    "labels": labels,
    "classes": classes,
    "dino_probs": dino_probs,
    "conv_probs": conv_probs,
    "ensemble_top1": fusion_result[0],
    "ensemble_top5": fusion_result[1],
    "fixed_weights": {
        "dino": 0.75,
        "conv": 0.25
    }
}, root / "final_food101_heldout_results.pth")

print("\nFinal results saved!")
print("Device used:", device)


In [ ]:

import time
import torch
import torch.nn as nn
import timm

from pathlib import Path
from PIL import Image
from torchvision import transforms as T

root = Path("/content/drive/MyDrive/NutriVision_AI")

# Restore both trained checkpoints
food_ckpt = torch.load(
    root / "dinov2_last4_best.pth",
    map_location="cpu",
    weights_only=True
)

nutrition_ckpt = torch.load(
    root / "nutrition_mlp_best.pth",
    map_location="cpu",
    weights_only=True
)

food_model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    num_classes=101,
    img_size=280
)

food_model.load_state_dict(
    food_ckpt["model_state_dict"], strict=True
)

class NutritionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(64, 6)
        )

    def forward(self, x):
        return self.net(x)

nutrition_model = NutritionMLP()
nutrition_model.load_state_dict(
    nutrition_ckpt["model_state_dict"], strict=True
)
nutrition_model.eval()

transform = T.Compose([
    T.Resize(320, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(280),
    T.ToTensor(),
    T.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    )
])

def predict_nutrivision(image, device="cpu"):
    food_model.to(device).eval()

    image = image.convert("RGB")
    x = transform(image).unsqueeze(0).to(device)

    start = time.perf_counter()

    with torch.inference_mode():
        if device == "cuda":
            with torch.autocast("cuda", dtype=torch.bfloat16):
                tokens = food_model.forward_features(x)
                embedding = food_model.forward_head(
                    tokens, pre_logits=True
                )
                logits = food_model.forward_head(tokens)
        else:
            tokens = food_model.forward_features(x)
            embedding = food_model.forward_head(
                tokens, pre_logits=True
            )
            logits = food_model.forward_head(tokens)

        probabilities = torch.softmax(
            logits.float(), dim=1
        )[0]

        # Use the same visual representation for calories
        features = embedding.float().cpu()

        normalized = (
            features - nutrition_ckpt["feature_mean"]
        ) / nutrition_ckpt["feature_std"]

        nutrition_output = nutrition_model(normalized)

        raw = (
            nutrition_output * nutrition_ckpt["target_std"]
            + nutrition_ckpt["target_mean"]
        )

        calories = torch.expm1(
            raw[:, 0].clamp(0, 9)
        ).clamp_min(0).item()

    elapsed = time.perf_counter() - start

    top_prob, top_idx = probabilities.max(dim=0)

    return {
        "food_name": food_ckpt["class_names"][top_idx.item()],
        "confidence": round(top_prob.item() * 100, 2),
        "estimated_calories": round(calories, 1),
        "inference_seconds": round(elapsed, 3),
        "device": device
    }

# Test using a Nutrition5k image already downloaded
results = torch.load(
    root / "nutrition_mlp_test_results.pth",
    map_location="cpu",
    weights_only=True
)

dish_id = results["test_ids"][0]

image_path = (
    Path("/content/data/nutrition5k/images")
    / f"{dish_id}.png"
)

assert image_path.exists(), f"Missing image: {image_path}"

image = Image.open(image_path)

print("===== CPU TEST =====")
cpu_result = predict_nutrivision(image, "cpu")
print(cpu_result)

print("\n===== CALORIE CONSISTENCY =====")
print(
    "Saved evaluation prediction:",
    round(float(results["predictions"][0]), 1), "kcal"
)
print(
    "Live image prediction:",
    cpu_result["estimated_calories"], "kcal"
)

if torch.cuda.is_available():
    print("\n===== GPU TEST =====")
    gpu_result = predict_nutrivision(image, "cuda")
    print(gpu_result)


In [ ]:

import time
import statistics
import torch

# Temporary warning threshold, not a validated OOD detector
CONFIDENCE_WARNING = 20.0

def safe_predict(image, device="cpu"):
    result = predict_nutrivision(image, device)

    if result["confidence"] < CONFIDENCE_WARNING:
        result["classification_status"] = "uncertain"
        result["message"] = (
            "Food recognition is uncertain. "
            "Please verify the food name."
        )
    else:
        result["classification_status"] = "predicted"
        result["message"] = None

    return result


def benchmark(image, device, repeats=10):
    # Warm up the model
    for _ in range(3):
        safe_predict(image, device)

    times = []

    for _ in range(repeats):
        if device == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()
        result = safe_predict(image, device)

        if device == "cuda":
            torch.cuda.synchronize()

        times.append(time.perf_counter() - start)

    print(f"\n===== {device.upper()} BENCHMARK =====")
    print("Median end-to-end function time:",
          round(statistics.median(times), 3), "seconds")
    print("Prediction:", result)


benchmark(image, "cpu")

if torch.cuda.is_available():
    benchmark(image, "cuda")


In [ ]:

import time
import statistics
import torch

# Temporary warning threshold, not a validated OOD detector
CONFIDENCE_WARNING = 20.0

def safe_predict(image, device="cpu"):
    result = predict_nutrivision(image, device)

    if result["confidence"] < CONFIDENCE_WARNING:
        result["classification_status"] = "uncertain"
        result["message"] = (
            "Food recognition is uncertain. "
            "Please verify the food name."
        )
    else:
        result["classification_status"] = "predicted"
        result["message"] = None

    return result


def benchmark(image, device, repeats=10):
    # Warm up the model
    for _ in range(3):
        safe_predict(image, device)

    times = []

    for _ in range(repeats):
        if device == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()
        result = safe_predict(image, device)

        if device == "cuda":
            torch.cuda.synchronize()

        times.append(time.perf_counter() - start)

    print(f"\n===== {device.upper()} BENCHMARK =====")
    print("Median end-to-end function time:",
          round(statistics.median(times), 3), "seconds")
    print("Prediction:", result)


benchmark(image, "cpu")

if torch.cuda.is_available():
    benchmark(image, "cuda")


In [ ]:

import torch
from IPython.display import display

display(image.resize((400, 400)))

device = next(food_model.parameters()).device

x = transform(image.convert("RGB")).unsqueeze(0).to(device)

food_model.eval()

with torch.inference_mode():
    logits = food_model(x)
    probs = torch.softmax(logits.float(), dim=1)[0]
    values, indices = probs.topk(5)

print("===== FOOD CLASSIFICATION TOP-5 =====")

for rank, (prob, idx) in enumerate(zip(values, indices), 1):
    name = food_ckpt["class_names"][idx.item()]
    print(f"{rank}. {name}: {prob.item()*100:.2f}%")

print("\nSum of all 101 probabilities:",
      round(probs.sum().item(), 4))


In [ ]:

import pandas as pd
from pathlib import Path
from PIL import Image
from IPython.display import display

data_dir = Path("/content/data/nutrition5k")

# The same image used in our previous test
dish_id = results["test_ids"][0]

metadata = pd.read_csv(data_dir / "metadata.csv")

row = metadata.loc[
    metadata["dish_id"].astype(str) == str(dish_id)
]

assert len(row) == 1, "Dish metadata not found"

print("===== ACTUAL DISH INFORMATION =====")
print("Dish ID:", dish_id)
print("Actual calories:", row["total_calories"].iloc[0])
print("Actual mass:", row["total_mass_g"].iloc[0], "g")
print("Ingredients:", row["ingredients"].iloc[0])

print("\n===== IMAGE =====")
display(
    Image.open(
        data_dir / "images" / f"{dish_id}.png"
    ).resize((450, 450))
)


In [ ]:

import io
import time
import torch
import pandas as pd

from PIL import Image, ImageOps
from google.colab import files
from IPython.display import display
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"

food_model = food_model.to(device).eval()
nutrition_model = nutrition_model.cpu().eval()

classes = food_ckpt["class_names"]

print("Upload food images for testing:")
uploaded = files.upload()

records = []

for filename, content in uploaded.items():

    image = Image.open(
        io.BytesIO(content)
    )

    image = ImageOps.exif_transpose(image).convert("RGB")

    x = transform(image).unsqueeze(0).to(device)

    if device == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():

        # Shared DINOv2 backbone
        tokens = food_model.forward_features(x)

        features = food_model.forward_head(
            tokens, pre_logits=True
        )

        logits = food_model.forward_head(tokens)

        probs = torch.softmax(
            logits.float(), dim=1
        )[0]

        top_probs, top_ids = probs.topk(3)

        # Nutrition MLP
        normalized = (
            features.float().cpu()
            - nutrition_ckpt["feature_mean"].float()
        ) / nutrition_ckpt["feature_std"].float()

        output = nutrition_model(normalized)

        raw = (
            output * nutrition_ckpt["target_std"].float()
            + nutrition_ckpt["target_mean"].float()
        )

        calories = torch.expm1(
            raw[:, 0].clamp(0, 9)
        ).clamp_min(0).item()

    if device == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    confidence = top_probs[0].item() * 100

    margin = (
        top_probs[0].item()
        - top_probs[1].item()
    ) * 100

    # Provisional warning rule, not validated OOD detection
    warning = confidence < 20 or margin < 5

    print("\n" + "=" * 45)
    print("IMAGE:", filename)

    display(image.resize((300, 300)))

    print("\nTop-3 Predictions:")

    for prob, idx in zip(top_probs, top_ids):
        print(
            f"  {classes[idx.item()]}: "
            f"{prob.item()*100:.2f}%"
        )

    print(f"\nEstimated Calories: {calories:.1f} kcal")
    print(f"Top-2 Margin: {margin:.2f} points")
    print(f"Inference Time: {elapsed:.3f} s")
    print(
        "Warning:",
        "Review food label" if warning else "No low-confidence alert"
    )

    records.append({
        "image": filename,
        "predicted_food": classes[top_ids[0].item()],
        "confidence_percent": round(confidence, 2),
        "top2_margin_points": round(margin, 2),
        "estimated_calories": round(calories, 1),
        "needs_review": warning,
        "inference_seconds": round(elapsed, 3)
    })

# Save results for project documentation
df = pd.DataFrame(records)

output_path = Path(
    "/content/drive/MyDrive/NutriVision_AI/"
    "real_world_test_results.csv"
)

df.to_csv(output_path, index=False)

print("\n===== REAL-WORLD TEST SUMMARY =====")
print("Images tested:", len(df))
print("Images flagged for review:", int(df["needs_review"].sum()))
print("Results saved:", output_path)


In [ ]:

import torch
from pathlib import Path
from PIL import Image
from transformers import pipeline

device_id = 0 if torch.cuda.is_available() else -1

# Open-vocabulary food recognition
recognizer = pipeline(
    "zero-shot-image-classification",
    model="google/siglip2-base-patch16-256",
    device=device_id
)

# Preserve all original Food-101 food names
known_foods = [
    name.replace("_", " ")
    for name in food_ckpt["class_names"]
]

# Add regional dishes and mixed meals
extra_foods = [
    "Saudi Arabian chicken kabsa with spiced rice",
    "Saudi Arabian lamb kabsa with rice",
    "chicken mandi with rice",
    "lamb mandi with rice",
    "chicken biryani",
    "grilled chicken with rice",
    "mixed vegetables on a plate",
    "vegetable salad",
    "grilled meat with rice",
    "roasted vegetables"
]

food_labels = list(dict.fromkeys(
    known_foods + extra_foods
))

# Ingredient candidates for mixed dishes
ingredient_labels = [
    "olives",
    "celery",
    "brussels sprouts",
    "chicken",
    "beef",
    "lamb",
    "rice",
    "potatoes",
    "tomatoes",
    "lettuce",
    "cucumber",
    "carrots",
    "broccoli",
    "cheese",
    "eggs",
    "beans",
    "peas",
    "fish",
    "shrimp"
]

def inspect_food(image_path):
    image = Image.open(image_path).convert("RGB")

    # Recognize the overall dish
    dish_results = recognizer(
        image,
        candidate_labels=food_labels,
        hypothesis_template="This is a photo of {}."
    )

    # Inspect the full image and overlapping crops
    w, h = image.size
    regions = [
        image,
        image.crop((0, 0, int(w*.7), int(h*.7))),
        image.crop((int(w*.3), 0, w, int(h*.7))),
        image.crop((0, int(h*.3), int(w*.7), h)),
        image.crop((int(w*.3), int(h*.3), w, h))
    ]

    ingredient_scores = {
        name: 0.0 for name in ingredient_labels
    }

    for region in regions:
        predictions = recognizer(
            region,
            candidate_labels=ingredient_labels,
            hypothesis_template="This is a photo of {}."
        )

        for p in predictions:
            ingredient_scores[p["label"]] = max(
                ingredient_scores[p["label"]],
                float(p["score"])
            )

    ingredients = sorted(
        ingredient_scores,
        key=ingredient_scores.get,
        reverse=True
    )

    print("\nIMAGE:", Path(image_path).name)
    print("Dish candidates:")
    for p in dish_results[:5]:
        print(" -", p["label"])

    print("Possible ingredients:")
    for name in ingredients[:5]:
        print(" -", name)

    print("These are suggestions, not verified labels.")

# Test the two images already used in our project
test_images = [
    Path("/content/كبسه (1)"),
    Path(
        "/content/data/nutrition5k/images/"
        "dish_1556575327.png"
    )
]

for path in test_images:
    if path.exists():
        inspect_food(path)
    else:
        print("Image not found:", path)


In [ ]:

!pip -q install fastapi uvicorn python-multipart


In [ ]:

%%writefile nutrivision_api.py

import io
import os
import time
import threading
from contextlib import asynccontextmanager, nullcontext
from pathlib import Path

import torch
import torch.nn as nn
import timm

from PIL import Image, ImageOps, UnidentifiedImageError
from torchvision import transforms as T
from transformers import pipeline

from fastapi import FastAPI, UploadFile, File, Form, HTTPException


MODEL_DIR = Path(os.getenv(
    "NUTRIVISION_MODEL_DIR",
    "/content/drive/MyDrive/NutriVision_AI"
))

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OPEN_VOCAB = os.getenv("NUTRIVISION_OPEN_VOCAB", "1") == "1"

EXTRA_FOODS = [
    "Saudi Arabian chicken kabsa with spiced rice",
    "Saudi Arabian lamb kabsa with rice",
    "chicken mandi with rice",
    "lamb mandi with rice",
    "chicken biryani",
    "grilled chicken with rice",
    "mixed vegetables on a plate",
    "vegetable salad",
    "grilled meat with rice",
    "roasted vegetables"
]

INGREDIENTS = [
    "olives", "celery", "brussels sprouts",
    "chicken", "beef", "lamb", "rice",
    "potatoes", "tomatoes", "lettuce",
    "cucumber", "carrots", "broccoli",
    "cheese", "eggs", "beans", "peas",
    "fish", "shrimp"
]


class NutritionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(64, 6)
        )

    def forward(self, x):
        return self.net(x)


class NutriVisionService:
    def __init__(self):
        print("Loading NutriVision AI...")
        print("Device:", DEVICE)

        food_ckpt = torch.load(
            MODEL_DIR / "dinov2_last4_best.pth",
            map_location="cpu",
            weights_only=True
        )

        nutrition_ckpt = torch.load(
            MODEL_DIR / "nutrition_mlp_best.pth",
            map_location="cpu",
            weights_only=True
        )

        self.classes = list(food_ckpt["class_names"])

        self.food_model = timm.create_model(
            "vit_base_patch14_dinov2.lvd142m",
            pretrained=False,
            num_classes=101,
            img_size=280
        )

        self.food_model.load_state_dict(
            food_ckpt["model_state_dict"],
            strict=True
        )
        self.food_model = self.food_model.to(DEVICE).eval()

        self.nutrition_model = NutritionMLP()
        self.nutrition_model.load_state_dict(
            nutrition_ckpt["model_state_dict"],
            strict=True
        )
        self.nutrition_model = (
            self.nutrition_model.to(DEVICE).eval()
        )

        self.feature_mean = (
            nutrition_ckpt["feature_mean"].float().to(DEVICE)
        )
        self.feature_std = (
            nutrition_ckpt["feature_std"].float().to(DEVICE)
        )
        self.target_mean = (
            nutrition_ckpt["target_mean"].float().to(DEVICE)
        )
        self.target_std = (
            nutrition_ckpt["target_std"].float().to(DEVICE)
        )

        self.transform = T.Compose([
            T.Resize(
                320,
                interpolation=T.InterpolationMode.BICUBIC
            ),
            T.CenterCrop(280),
            T.ToTensor(),
            T.Normalize(
                (0.485, 0.456, 0.406),
                (0.229, 0.224, 0.225)
            )
        ])

        self.recognizer = None

        if OPEN_VOCAB:
            print("Loading SigLIP2...")
            self.recognizer = pipeline(
                "zero-shot-image-classification",
                model="google/siglip2-base-patch16-256",
                device=0 if DEVICE.type == "cuda" else -1
            )

        self.dish_candidates = list(dict.fromkeys(
            [c.replace("_", " ") for c in self.classes]
            + EXTRA_FOODS
        ))

        print("NutriVision AI ready!")

    def predict(self, image, analyze_ingredients=True):

        start = time.perf_counter()

        image = ImageOps.exif_transpose(image).convert("RGB")
        x = self.transform(image).unsqueeze(0).to(DEVICE)

        amp = (
            torch.autocast("cuda", dtype=torch.bfloat16)
            if DEVICE.type == "cuda"
            else nullcontext()
        )

        with torch.inference_mode():
            with amp:
                tokens = self.food_model.forward_features(x)

                features = self.food_model.forward_head(
                    tokens, pre_logits=True
                )

                logits = self.food_model.forward_head(tokens)

            probabilities = torch.softmax(
                logits.float(), dim=1
            )[0]

            values, indices = probabilities.topk(3)

            normalized = (
                features.float() - self.feature_mean
            ) / self.feature_std

            output = self.nutrition_model(normalized)

            raw = (
                output * self.target_std
                + self.target_mean
            )

            calories = torch.expm1(
                raw[:, 0].clamp(0, 9)
            ).clamp_min(0).item()

        food_top3 = [
            {
                "name": self.classes[int(idx)],
                "softmax_score_percent": round(
                    float(prob) * 100, 2
                )
            }
            for prob, idx in zip(values, indices)
        ]

        confidence = food_top3[0]["softmax_score_percent"]
        margin = float((values[0] - values[1]) * 100)

        dish_suggestions = []
        ingredient_suggestions = []

        if self.recognizer is not None:
            dish_results = self.recognizer(
                image,
                candidate_labels=self.dish_candidates,
                hypothesis_template="This is a photo of {}."
            )

            dish_suggestions = [
                item["label"]
                for item in dish_results[:5]
            ]

            if analyze_ingredients:
                w, h = image.size

                regions = [
                    image,
                    image.crop((0, 0, int(w*.7), int(h*.7))),
                    image.crop((int(w*.3), 0, w, int(h*.7))),
                    image.crop((0, int(h*.3), int(w*.7), h)),
                    image.crop((int(w*.3), int(h*.3), w, h))
                ]

                crop_results = self.recognizer(
                    regions,
                    candidate_labels=INGREDIENTS,
                    hypothesis_template="This is a photo of {}.",
                    batch_size=5
                )

                ingredient_scores = {
                    name: 0.0 for name in INGREDIENTS
                }

                for predictions in crop_results:
                    for item in predictions:
                        name = item["label"]
                        ingredient_scores[name] = max(
                            ingredient_scores[name],
                            float(item["score"])
                        )

                ingredient_suggestions = sorted(
                    ingredient_scores,
                    key=ingredient_scores.get,
                    reverse=True
                )[:5]

        primary = (
            dish_suggestions[0]
            if dish_suggestions
            else food_top3[0]["name"]
        )

        # Temporary review heuristic, not validated OOD detection
        needs_confirmation = (
            confidence < 50
            or margin < 20
            or primary in EXTRA_FOODS
        )

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

        return {
            "suggested_food": primary,
            "food101_top3": food_top3,
            "open_vocab_suggestions": dish_suggestions,
            "possible_ingredients": ingredient_suggestions,
            "estimated_calories": round(calories, 1),
            "calorie_unit": "kcal",
            "requires_confirmation": needs_confirmation,
            "user_can_correct_food": True,
            "message": (
                "Suggestions are not verified. "
                "Confirm the dish and ingredients. "
                "Calorie estimation may be less reliable "
                "for unfamiliar dishes and portion sizes."
            ),
            "device": str(DEVICE),
            "inference_seconds": round(elapsed, 3)
        }


service = None
inference_lock = threading.Lock()


@asynccontextmanager
async def lifespan(app):
    global service
    service = NutriVisionService()
    yield
    service = None


app = FastAPI(
    title="NutriVision AI API",
    version="0.1.0",
    lifespan=lifespan
)


@app.get("/health")
def health():
    return {
        "status": "ready" if service is not None else "loading",
        "device": str(DEVICE),
        "open_vocabulary_enabled": OPEN_VOCAB
    }


@app.post("/predict")
def predict(
    image: UploadFile = File(...),
    analyze_ingredients: bool = Form(True)
):
    content = image.file.read(10 * 1024 * 1024 + 1)

    if len(content) > 10 * 1024 * 1024:
        raise HTTPException(413, "Image exceeds 10 MB")

    try:
        with Image.open(io.BytesIO(content)) as img:
            if img.width * img.height > 20_000_000:
                raise HTTPException(413, "Image too large")
            uploaded_image = img.copy()

    except (UnidentifiedImageError, OSError, ValueError):
        raise HTTPException(400, "Invalid image file")

    with inference_lock:
        return service.predict(
            uploaded_image,
            analyze_ingredients=analyze_ingredients
        )


In [ ]:

import subprocess
import time
import requests
from pathlib import Path

# Run the API locally inside Colab
server = subprocess.Popen([
    "uvicorn",
    "nutrivision_api:app",
    "--host", "127.0.0.1",
    "--port", "8000"
])

# Wait until model loading has completed
for attempt in range(120):
    if server.poll() is not None:
        raise RuntimeError("API process stopped during startup")

    try:
        response = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=2
        )

        if response.ok and response.json()["status"] == "ready":
            break
    except requests.RequestException:
        pass

    time.sleep(2)
else:
    raise TimeoutError("API startup not completed")

print("===== API STATUS =====")
print(response.json())

# Test one of our existing food images
image_path = Path("/content/كبسه (1)")

if not image_path.exists():
    image_path = Path(
        "/content/data/nutrition5k/images/"
        "dish_1556575327.png"
    )

assert image_path.exists()

with open(image_path, "rb") as f:
    result = requests.post(
        "http://127.0.0.1:8000/predict",
        files={"image": (image_path.name, f, "image/jpeg")},
        data={"analyze_ingredients": "true"},
        timeout=180
    )

result.raise_for_status()

print("\n===== NUTRIVISION API RESULT =====")
print(result.json())


In [ ]:

import requests
from pathlib import Path

base_url = "http://127.0.0.1:8000"

test_images = {
    "Hamburger": "/content/burger.jpg",
    "Kabsa": "/content/كبسه (1)",
    "Mixed Vegetables":
        "/content/data/nutrition5k/images/dish_1556575327.png"
}

print("===== NUTRIVISION INTEGRATION TEST =====")

for name, filepath in test_images.items():
    path = Path(filepath)

    if not path.exists():
        print(f"{name}: IMAGE NOT FOUND")
        continue

    with open(path, "rb") as file:
        response = requests.post(
            f"{base_url}/predict",
            files={"image": (path.name, file, "image/jpeg")},
            data={"analyze_ingredients": "true"},
            timeout=180
        )

    response.raise_for_status()
    result = response.json()

    print(f"\n===== {name} =====")
    print("Suggested food:", result["suggested_food"])
    print("Food-101 prediction:", result["food101_top3"][0]["name"])
    print("Possible ingredients:", result["possible_ingredients"])
    print("Estimated calories:", result["estimated_calories"])
    print("Requires confirmation:", result["requires_confirmation"])
    print("Inference seconds:", result["inference_seconds"])


In [ ]:

%%writefile feedback_store.py

import os
import json
import uuid
import sqlite3

from pathlib import Path
from fastapi import APIRouter, HTTPException
from pydantic import BaseModel, Field

DB_PATH = Path(os.getenv(
    "NUTRIVISION_FEEDBACK_DB",
    "/content/drive/MyDrive/NutriVision_AI/feedback.sqlite3"
))

router = APIRouter()


def connect():
    return sqlite3.connect(DB_PATH, timeout=30)


def init_db():
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)

    with connect() as db:
        db.execute("""
            CREATE TABLE IF NOT EXISTS predictions (
                prediction_id TEXT PRIMARY KEY,
                result_json TEXT NOT NULL,
                corrected_food TEXT,
                confirmed_ingredients TEXT,
                created_at TEXT DEFAULT CURRENT_TIMESTAMP,
                corrected_at TEXT
            )
        """)


def register_prediction(result):
    prediction_id = str(uuid.uuid4())

    with connect() as db:
        db.execute(
            """
            INSERT INTO predictions
            (prediction_id, result_json)
            VALUES (?, ?)
            """,
            (
                prediction_id,
                json.dumps(result, ensure_ascii=False)
            )
        )

    return {
        **result,
        "prediction_id": prediction_id
    }


class FoodCorrection(BaseModel):
    prediction_id: str
    corrected_food: str = Field(min_length=1, max_length=150)
    confirmed_ingredients: list[str] = Field(
        default_factory=list
    )


@router.post("/feedback")
def save_feedback(feedback: FoodCorrection):
    with connect() as db:
        row = db.execute(
            "SELECT prediction_id FROM predictions "
            "WHERE prediction_id = ?",
            (feedback.prediction_id,)
        ).fetchone()

        if row is None:
            raise HTTPException(
                status_code=404,
                detail="Prediction not found"
            )

        db.execute(
            """
            UPDATE predictions
            SET corrected_food = ?,
                confirmed_ingredients = ?,
                corrected_at = CURRENT_TIMESTAMP
            WHERE prediction_id = ?
            """,
            (
                feedback.corrected_food,
                json.dumps(
                    feedback.confirmed_ingredients,
                    ensure_ascii=False
                ),
                feedback.prediction_id
            )
        )

    return {
        "saved": True,
        "prediction_id": feedback.prediction_id,
        "corrected_food": feedback.corrected_food
    }


In [ ]:
from feedback_store import router, init_db, register_prediction

In [ ]:
with inference_lock:
    result = service.predict(
        uploaded_image,
        analyze_ingredients=analyze_ingredients
    )

return register_prediction(result)

In [ ]:

from pathlib import Path

api_path = Path("/content/nutrivision_api.py")
feedback_path = Path("/content/feedback_store.py")

assert api_path.exists(), "API file not found"
assert feedback_path.exists(), "Run feedback_store.py creation cell first"

code = api_path.read_text(encoding="utf-8")

# 1. Import feedback functions
new_import = (
    "from feedback_store import "
    "router, init_db, register_prediction"
)

if new_import not in code:
    anchor = "from fastapi import FastAPI, UploadFile, File, Form, HTTPException"
    assert anchor in code
    code = code.replace(anchor, anchor + "\n" + new_import, 1)

# 2. Initialize database on startup
old = "    global service\n    service = NutriVisionService()"
new = "    global service\n    init_db()\n    service = NutriVisionService()"

if "    init_db()" not in code:
    assert old in code
    code = code.replace(old, new, 1)

# 3. Register feedback endpoint
if "app.include_router(router)" not in code:
    anchor = '\n\n@app.get("/health")'
    assert anchor in code
    code = code.replace(
        anchor,
        "\n\napp.include_router(router)" + anchor,
        1
    )

# 4. Save predictions and return prediction_id
old = """    with inference_lock:
        return service.predict(
            uploaded_image,
            analyze_ingredients=analyze_ingredients
        )"""

new = """    with inference_lock:
        result = service.predict(
            uploaded_image,
            analyze_ingredients=analyze_ingredients
        )

    return register_prediction(result)"""

if old in code:
    code = code.replace(old, new, 1)
else:
    assert "return register_prediction(result)" in code, (
        "Could not find the original predict endpoint"
    )

api_path.write_text(code, encoding="utf-8")

print("===== API UPDATE =====")
print("Feedback import: OK")
print("Database initialization: OK")
print("Feedback endpoint: OK")
print("Prediction ID integration: OK")
print("File updated successfully!")


In [ ]:

import subprocess
import requests
import time

# Stop previous server
if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=15)

# Start updated version
server = subprocess.Popen(
    [
        "uvicorn",
        "nutrivision_api:app",
        "--host", "127.0.0.1",
        "--port", "8000"
    ],
    cwd="/content",
    stdout=open("/content/nutrivision_api.log", "w"),
    stderr=subprocess.STDOUT
)

for _ in range(120):
    if server.poll() is not None:
        raise RuntimeError(
            Path("/content/nutrivision_api.log").read_text()[-3000:]
        )

    try:
        r = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=2
        )
        if r.ok and r.json()["status"] == "ready":
            print("API READY:", r.json())
            break
    except requests.RequestException:
        pass

    time.sleep(2)
else:
    raise TimeoutError("API startup timed out")


In [ ]:

import requests

url = "http://127.0.0.1:8000"

with open("/content/كبسه (1)", "rb") as f:
    response = requests.post(
        url + "/predict",
        files={"image": ("kabsa.jpg", f, "image/jpeg")},
        timeout=180
    )

response.raise_for_status()
result = response.json()

print("Prediction ID:", result["prediction_id"])
print("Suggested Food:", result["suggested_food"])

feedback = requests.post(
    url + "/feedback",
    json={
        "prediction_id": result["prediction_id"],
        "corrected_food": "Chicken Kabsa",
        "confirmed_ingredients": ["chicken", "rice"]
    },
    timeout=30
)

feedback.raise_for_status()

print("\n===== FEEDBACK RESULT =====")
print(feedback.json())
